In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
rocket_global_allocator_expr_v3_SAMEDAY_RR_POLICY_WR_THROTTLE.py

Based on v2_SAMEDAY_RR_POLICY, with NEW:

D) Rolling wR (risk-weighted R) fine-grained gate with RISK THROTTLE (not kill-switch)
   - Gate key includes RR: (regime, side, side_mode, stop_r, tgt_r)
   - Uses CLOSED trades (realized) in a rolling window (exit_day-based; causal for next-day entries)
   - If rolling wR is bad for enough consecutive eval-days => risk multiplier set to THROTTLE_MULT
   - If rolling wR recovers for enough consecutive eval-days => risk multiplier resets to 1.0
   - Gate affects position sizing: risk_amt = base_risk_amt * risk_mult(key)
   - Optional score penalty for throttled keys (default 0.0) to avoid “crowding” into throttled slices

Outputs (same as v2) plus:
- global/wr_gate_daily_model.csv
- global/wr_gate_events_model.csv
- global/wr_gate_state_end_model.csv

No argparse. Edit CONFIG.
"""

import os
import re
import sys
import json
import time
import zipfile
import logging
import hashlib
from dataclasses import dataclass
from datetime import datetime
from typing import Dict, Tuple, List, Optional
from collections import deque, Counter

import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.linear_model import RidgeClassifier, LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier


# =========================================================
# CONFIG (EDIT THESE)
# =========================================================

BASELINE_RESULTS_PATH = r"/kaggle/input/datasets/auriniumus/rsi-divergence-portfolio-144"
OHLCV_FLAGS_CSV = r"/kaggle/input/datasets/auriniumus/ohlcv-with-rsi-divergence-flags-from-kite-1-csv/ohlcv_with_rsi_divergence_flags_from_kite (1).csv"

OUT_DIR = "/kaggle/working/global_allocator_expr_v3" if os.path.isdir("/kaggle/working") else "/mnt/data/global_allocator_expr_v3"

EXCLUDE_ETF = True

EVAL_START = "2015-01-01"
EVAL_END   = "2025-12-10"

SUSP_GAP_DAYS = 7

ROLLING_LOOKBACK_YEARS = 5
ROLLING_FIRST_TEST_YEAR = None
ROLLING_MIN_TEST_DAYS = 60

WINNER_CAGR_MIN = 0.0
WINNER_MIN_TRADES = 800
WINNER_MIN_EQUITY_DAYS = 150

ROCKET_WINDOWS = [7, 15, 21]
ROCKET_CHANNELS = ["log_ret_close", "hl_range", "oc_move", "vol_chg"]

ROCKET_CFG = dict(kernels={7: 512, 15: 512, 21: 1024}, max_dilation=4)
ROCKET_RANDOM_SEED = 4242
ROCKET_BATCH_SIZE = 5000

ADD_STATIC_FEATURES = True

VALID_FRAC = 0.20
VALID_DAYS_MAX = 90
VALID_DAYS_MIN = 20
MIN_TRAIN_ROWS = 2000
MIN_VAL_ROWS   = 1000

MAX_TRAIN_SAMPLES_PER_BUCKET = 160_000
MAX_VAL_SAMPLES_PER_BUCKET   = 60_000

RIDGE_ALPHA = 1.0
RF_PWIN_PARAMS = dict(
    n_estimators=400,
    max_depth=14,
    min_samples_leaf=25,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42,
)

MAG_BIN_QUANTILES = [0.50, 0.80, 0.95, 0.99]
MAG_RF_PARAMS = dict(
    n_estimators=400,
    max_depth=14,
    min_samples_leaf=20,
    max_features="sqrt",
    n_jobs=-1,
    random_state=123,
)

R_ABS_MAX_SANITY = 20.0

SELECTION_MODE = "top_k_per_day"   # or "expr_threshold"
TOP_K_PER_DAY = 80                # kept for compatibility (cap_entries governs actual cap)
EXPR_THRESHOLD = 0.0              # used only if SELECTION_MODE == "expr_threshold"

WINNER_RESERVE_FRAC = 0.60
WINNER_COVERAGE_FLOOR = 0.70
WINNER_UNDERCOVER_BONUS = 0.15

ENFORCE_ONE_RR_PER_SIGNAL = True

START_EQUITY = 1_000_000.0
RISK_PCT_PER_TRADE = 0.005
MAX_OPEN_POSITIONS = 180
MAX_GROSS_RISK_PCT = 0.35
MAX_NEW_ENTRIES_PER_DAY = 120

# ---- Replay / diagnostics ----
REINDEX_EQUITY_TO_BUSINESS_DAYS = True
REPORT_TERMINAL_SOURCE_PNL = True   # report closed-book summary for leftover open positions using source trade PnL
WRITE_DEBUG_FILTER_COUNTS = True
WRITE_DEBUG_OPEN_MARKS = True

W_PWIN_RIDGE = 0.50
W_PWIN_RF    = 0.50

FEATURE_DTYPE = np.float16
TRAIN_DTYPE = np.float32
PRED_BATCH = 60_000

# ---- Missing bucket model handling ----
MISSING_BUCKET_FALLBACK = "baseline"   # "baseline" or "drop"
SENTINEL_SCORE = -1e9

# ---- RR policy ----
RR_POLICY_ENABLE = True
RR_POLICY_BUCKET_AWARE = True
RR_POLICY_GLOBAL_FALLBACK = True
RR_POLICY_RIDGE_ALPHA = 100.0
RR_POLICY_RIDGE_SOLVER = "lsqr"
RR_POLICY_MIN_ROWS_PER_PAIR = 2500
RR_POLICY_MAX_ROWS_PER_PAIR = 120_000
RR_POLICY_BASELINE_CAGR_WEIGHT = 0.25
RR_POLICY_PRED_CLIP = R_ABS_MAX_SANITY

# ---- NEW: Rolling wR THROTTLE gate (fine key includes RR) ----
WR_GATE_ENABLE_MODEL = True
WR_GATE_ENABLE_BASELINE = False

# Fine key: (regime, side, side_mode, stop_r, tgt_r)
WR_GATE_WINDOW_DAYS = 60
WR_GATE_MIN_TRADES = 60        # lower this to trigger earlier (e.g., 30–80)
WR_GATE_MIN_DAYS = 20          # avoid flipping on tiny time span
WR_GATE_USE_RISK_WEIGHT = True # wR uses risk_amt as weight

WR_GATE_KILL_TH = -0.05        # if wR <= this for KILL_STREAK days => throttle
WR_GATE_REVIVE_TH = +0.05      # if wR >= this for REVIVE_STREAK days => unthrottle
WR_GATE_KILL_STREAK = 3
WR_GATE_REVIVE_STREAK = 5

WR_GATE_THROTTLE_MULT = 0.25   # risk multiplier when throttled
WR_GATE_SCORE_PENALTY = 0.00   # optional: subtract this from score to downrank throttled keys (0.0 = no ranking effect)


# =========================================================
# LOGGING
# =========================================================
os.makedirs(OUT_DIR, exist_ok=True)
for sub in ["caches", "features", "folds", "global", "models"]:
    os.makedirs(os.path.join(OUT_DIR, sub), exist_ok=True)

LOG_PATH = os.path.join(OUT_DIR, "run.log")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.FileHandler(LOG_PATH, mode="w", encoding="utf-8"),
              logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger("global_allocator_expr_v3_wr_throttle")


# =========================================================
# CANONICAL DATE HELPERS (datetime64[ns] normalized)
# =========================================================
def s_day(series: pd.Series, fmt: Optional[str] = None, default_dayfirst: bool = False) -> pd.Series:
    if fmt is not None:
        dt = pd.to_datetime(series, format=fmt, errors="coerce", utc=True)
    else:
        dt = pd.to_datetime(series, errors="coerce", dayfirst=default_dayfirst, utc=True)
    dt = dt.dt.tz_convert(None)
    return dt.dt.normalize()

def assert_day_dt(df: pd.DataFrame, col: str, name: str = ""):
    if col not in df.columns:
        raise RuntimeError(f"[ASSERT] missing column: {col}")
    if not np.issubdtype(df[col].dtype, np.datetime64):
        raise RuntimeError(f"[ASSERT] {name}{col} must be datetime64[ns]. Got {df[col].dtype}")
    bad = (df[col].dt.hour != 0) | (df[col].dt.minute != 0) | (df[col].dt.second != 0) | (df[col].dt.nanosecond != 0)
    if bad.any():
        raise RuntimeError(f"[ASSERT] {name}{col} contains non-midnight times.")


# =========================================================
# DATE INFERENCE (OHLCV)
# =========================================================
_MONTH_RE = re.compile(r"[A-Za-z]")

def _date_part(s: str) -> str:
    s = str(s).strip()
    if not s:
        return s
    if "T" in s:
        s = s.split("T", 1)[0]
    if " " in s:
        s = s.split(" ", 1)[0]
    return s.strip()

def _tokenize_numeric_date(dp: str):
    dp = dp.strip()
    if not dp:
        return None, None
    for sep in ("-", "/", "."):
        if sep in dp:
            toks = [t.strip() for t in dp.split(sep) if t.strip() != ""]
            return sep, toks
    return None, None

def infer_date_format_from_samples(samples, default_dayfirst=False):
    cleaned = []
    for x in samples:
        if x is None or (isinstance(x, float) and np.isnan(x)):
            continue
        s = str(x).strip()
        if not s:
            continue
        cleaned.append(_date_part(s))
    cleaned = [c for c in cleaned if c]
    if not cleaned:
        return None, default_dayfirst, "no_samples"

    if any(_MONTH_RE.search(c) for c in cleaned[:200]):
        month_formats = [
            "%d-%b-%Y", "%d-%b-%y", "%d-%B-%Y", "%d-%B-%y",
            "%b-%d-%Y", "%b-%d-%y", "%B-%d-%Y", "%B-%d-%y",
        ]
        best, best_ok = None, -1
        probe = cleaned[:200]
        for fmt in month_formats:
            ok = 0
            for v in probe:
                try:
                    datetime.strptime(v, fmt)
                    ok += 1
                except Exception:
                    pass
            if ok > best_ok:
                best_ok = ok
                best = fmt
        if best_ok > 0:
            return best, default_dayfirst, f"month_name_fmt_ok={best_ok}/{len(probe)}"
        return None, default_dayfirst, "month_name_unrecognized"

    sep = None
    token_lists = []
    for v in cleaned[:500]:
        ssep, toks = _tokenize_numeric_date(v)
        if ssep is None or toks is None:
            continue
        if sep is None:
            sep = ssep
        if ssep != sep:
            continue
        if len(toks) == 3:
            token_lists.append(toks)
    if not token_lists or sep is None:
        return None, default_dayfirst, "numeric_unrecognized"

    year_pos_votes = {0: 0, 2: 0}
    year_is_2digit_votes = 0
    for toks in token_lists[:200]:
        t0, _, t2 = toks
        if len(t0) == 4 and t0.isdigit():
            year_pos_votes[0] += 2
        if len(t2) == 4 and t2.isdigit():
            year_pos_votes[2] += 2
        if len(t2) == 2 and t2.isdigit():
            year_is_2digit_votes += 1

    year_pos = 0 if year_pos_votes[0] > year_pos_votes[2] else 2
    year_fmt = "%Y"
    if year_pos == 2 and year_is_2digit_votes > 0:
        year_fmt = "%y"

    decided = None
    for toks in token_lists[:500]:
        try:
            if year_pos == 2:
                a, b = int(toks[0]), int(toks[1])
            else:
                a, b = int(toks[1]), int(toks[2])
        except Exception:
            continue
        if b > 12 and a <= 12:
            decided = "md"
            break
        if a > 12 and b <= 12:
            decided = "dm"
            break
    if decided is None:
        decided = "dm" if default_dayfirst else "md"

    if year_pos == 2:
        return (f"%d{sep}%m{sep}{year_fmt}", True, "year_last_dayfirst") if decided == "dm" else (f"%m{sep}%d{sep}{year_fmt}", False, "year_last_monthfirst")
    else:
        return (f"%Y{sep}%d{sep}%m", True, "year_first_ydm") if decided == "dm" else (f"%Y{sep}%m{sep}%d", False, "year_first_ymd")

def parse_date_series_with_inference(series: pd.Series, default_dayfirst=False, sample_n=300):
    s = series.astype(str).map(lambda x: x.strip() if isinstance(x, str) else str(x))
    s = s.replace("nan", "").replace("None", "")
    s_date = s.map(_date_part)
    non_empty = [v for v in s_date.head(sample_n).tolist() if v and v.lower() not in {"nan", "none"}]
    fmt, dayfirst_used, note = infer_date_format_from_samples(non_empty, default_dayfirst=default_dayfirst)
    if fmt is not None:
        dt = s_day(s_date, fmt=fmt, default_dayfirst=dayfirst_used)
        return dt, fmt, note
    dt1 = s_day(s_date, fmt="%Y-%m-%d", default_dayfirst=False)
    if dt1.notna().mean() > 0.9:
        return dt1, "%Y-%m-%d", "fallback_iso"
    dt2 = s_day(s_date, fmt=None, default_dayfirst=default_dayfirst)
    return dt2, None, f"fallback_pandas_dayfirst={default_dayfirst}"


# =========================================================
# UTILS
# =========================================================
def _is_etf_symbol(sym):
    s = str(sym).upper()
    return ("ETF" in s) or ("BEES" in s)

def pick_val_dates(train_dates: np.ndarray) -> set:
    n = len(train_dates)
    if n <= 0:
        return set()
    vd = int(round(n * float(VALID_FRAC)))
    vd = max(VALID_DAYS_MIN, min(VALID_DAYS_MAX, vd))
    vd = min(vd, max(10, n // 2))
    return set(train_dates[-vd:])

def normalize_kernels_dict(kernels) -> Dict[int, int]:
    return {int(k): int(v) for k, v in dict(kernels).items()}

def normalize_rocket_cfg(cfg: dict) -> dict:
    cfg2 = dict(cfg)
    cfg2["kernels"] = normalize_kernels_dict(cfg2["kernels"])
    cfg2["max_dilation"] = int(cfg2["max_dilation"])
    return cfg2

def _stable_symbol_hash(symbols_needed) -> str:
    syms = sorted({str(s).upper().strip() for s in symbols_needed})
    h = hashlib.sha1()
    for s in syms:
        h.update(s.encode("utf-8", errors="ignore"))
        h.update(b"\n")
    return h.hexdigest()[:16]

def build_ohlcv_cache_paths(csv_path: str, symbols_needed) -> Tuple[str, str, str]:
    st = os.stat(csv_path)
    base = os.path.splitext(os.path.basename(csv_path))[0]
    sym_hash = _stable_symbol_hash(symbols_needed)
    tag = f"{base}_gap{SUSP_GAP_DAYS}_n{len(set(symbols_needed))}_sz{int(st.st_size)}_mt{int(st.st_mtime_ns)}_h{sym_hash}"
    cache_series = os.path.join(OUT_DIR, "caches", f"ohlcv_series_{tag}.pkl")
    cache_seg = os.path.join(OUT_DIR, "caches", f"ohlcv_seg_{tag}.pkl")
    cache_meta = os.path.join(OUT_DIR, "caches", f"ohlcv_meta_{tag}.json")
    return cache_series, cache_seg, cache_meta

def subsample_stratified(df: pd.DataFrame, ycol: str, nmax: int, seed: int) -> pd.DataFrame:
    if len(df) <= nmax:
        return df
    rng = np.random.RandomState(seed)
    y = df[ycol].astype(int).values
    idx0 = np.where(y == 0)[0]
    idx1 = np.where(y == 1)[0]
    if idx0.size == 0 or idx1.size == 0:
        take = rng.choice(np.arange(len(df)), size=nmax, replace=False)
        return df.iloc[take].copy()
    n1 = min(idx1.size, nmax // 2)
    n0 = min(idx0.size, nmax - n1)
    take = np.concatenate([
        rng.choice(idx1, size=n1, replace=False),
        rng.choice(idx0, size=n0, replace=False),
    ])
    rng.shuffle(take)
    return df.iloc[take].copy()

def subsample_random(df: pd.DataFrame, nmax: int, seed: int) -> pd.DataFrame:
    if len(df) <= nmax:
        return df
    rng = np.random.RandomState(seed)
    take = rng.choice(df.index.values, size=nmax, replace=False)
    return df.loc[take].copy()


# =========================================================
# BASELINE FILES (portfolio_run)
# =========================================================
def resolve_root_dir(path_in, extract_to_dir):
    if os.path.isdir(path_in):
        return path_in
    if os.path.isfile(path_in) and path_in.lower().endswith(".zip"):
        if os.path.isdir(extract_to_dir) and any(os.scandir(extract_to_dir)):
            return extract_to_dir
        os.makedirs(extract_to_dir, exist_ok=True)
        with zipfile.ZipFile(path_in, "r") as zf:
            zf.extractall(extract_to_dir)
        return extract_to_dir
    raise FileNotFoundError(f"Path must be a directory or a .zip file. Got: {path_in}")

def locate_portfolio_run_dir(root_dir):
    cand = os.path.join(root_dir, "portfolio_run")
    if os.path.isdir(cand):
        return cand
    for r, _d, _f in os.walk(root_dir):
        if os.path.basename(r).lower() == "portfolio_run":
            return r
    raise FileNotFoundError(f"Could not locate 'portfolio_run' under: {root_dir}")

def locate_trades_dir(portfolio_run_dir):
    cand = os.path.join(portfolio_run_dir, "trades")
    if os.path.isdir(cand):
        return cand
    for r, _d, _f in os.walk(portfolio_run_dir):
        if os.path.basename(r).lower() == "trades":
            return r
    raise FileNotFoundError(f"Could not locate 'trades' under: {portfolio_run_dir}")

def locate_equity_dir(portfolio_run_dir):
    cand = os.path.join(portfolio_run_dir, "equity_curves")
    if os.path.isdir(cand):
        return cand
    for r, _d, _f in os.walk(portfolio_run_dir):
        if os.path.basename(r).lower() in {"equity_curves", "equity"}:
            return r
    raise FileNotFoundError(f"Could not locate 'equity_curves' under: {portfolio_run_dir}")

_RE_EQ = re.compile(r"^equity_(bull|bear)(?:_(both|long|short))?_s([0-9_.]+)_t([0-9_.]+)\.csv$", re.IGNORECASE)
_RE_TR = re.compile(r"^trades_(bull|bear)(?:_(both|long|short))?_s([0-9_.]+)_t([0-9_.]+)\.(parquet|csv)$", re.IGNORECASE)

def _num_to_float(s: str) -> float:
    return float(str(s).replace("_", "."))

def load_baseline_equity_curves(baseline_equity_dir: str):
    eq_map: Dict[Tuple[str, str, float, float], pd.DataFrame] = {}
    files = sorted([f for f in os.listdir(baseline_equity_dir) if f.lower().endswith(".csv")])
    if not files:
        raise RuntimeError(f"No equity curve CSV files found in {baseline_equity_dir}")

    matched = 0
    for fn in files:
        m = _RE_EQ.match(fn)
        if not m:
            continue
        matched += 1
        regime = m.group(1).lower()
        side_mode = (m.group(2) or "both").lower()
        stop = _num_to_float(m.group(3))
        tgt  = _num_to_float(m.group(4))
        fp = os.path.join(baseline_equity_dir, fn)

        df = pd.read_csv(fp)
        df.columns = [c.strip() for c in df.columns]
        if "Date" not in df.columns:
            col_date = next((c for c in df.columns if c.lower() == "date"), None)
            if col_date:
                df.rename(columns={col_date: "Date"}, inplace=True)
        if "Equity" not in df.columns:
            col_eq = next((c for c in df.columns if c.lower() == "equity"), None)
            if col_eq:
                df.rename(columns={col_eq: "Equity"}, inplace=True)
        if "Date" not in df.columns or "Equity" not in df.columns:
            continue

        df["date"] = s_day(df["Date"], fmt=None, default_dayfirst=False)
        df["equity"] = pd.to_numeric(df["Equity"], errors="coerce")
        df = df.dropna(subset=["date", "equity"]).sort_values("date")
        if df.empty:
            continue

        assert_day_dt(df, "date", name="[BASELINE_EQ] ")
        key = (regime, side_mode, float(stop), float(tgt))
        eq_map[key] = df[["date", "equity"]].copy()

    if not eq_map:
        raise RuntimeError("No equity curves loaded.")
    log.info(f"[BASELINE] equity_curves: matched={matched:,} loaded={len(eq_map):,}")
    return eq_map

def _parse_key_from_srcfile(srcfile: str):
    base = os.path.basename(srcfile)
    m = _RE_TR.match(base)
    if not m:
        return None
    regime = m.group(1).lower()
    side_mode = (m.group(2) or "both").lower()
    stop = _num_to_float(m.group(3))
    tgt  = _num_to_float(m.group(4))
    return regime, side_mode, stop, tgt

def load_all_trades_from_dir(trades_dir: str):
    parqs = [os.path.join(trades_dir, f) for f in os.listdir(trades_dir) if f.lower().endswith(".parquet")]
    csvs  = [os.path.join(trades_dir, f) for f in os.listdir(trades_dir) if f.lower().endswith(".csv")]
    if not parqs and not csvs:
        raise RuntimeError(f"No trade files found in {trades_dir}")

    dfs = []
    if parqs:
        log.info(f"[TRADES] Reading parquet files: {len(parqs)}")
        for p in sorted(parqs):
            df = pd.read_parquet(p)
            df["__srcfile"] = os.path.basename(p)
            dfs.append(df)
    else:
        log.info(f"[TRADES] Reading csv files: {len(csvs)}")
        for p in sorted(csvs):
            df = pd.read_csv(p)
            df["__srcfile"] = os.path.basename(p)
            dfs.append(df)

    t = pd.concat(dfs, ignore_index=True)
    t.columns = [c.strip() for c in t.columns]

    def pick(names):
        for n in names:
            if n in t.columns:
                return n
        return None

    c_entry = pick(["EntryDate", "entry_date", "entry_dt"])
    c_exit  = pick(["ExitDate", "exit_date", "exit_dt"])
    c_sym   = pick(["Symbol", "symbol", "Ticker", "ticker", "tradingsymbol", "TradingSymbol"])
    c_r     = pick(["Rmult", "rmult", "RMultiple"])
    c_reg   = pick(["Regime", "regime"])
    c_side  = pick(["Side", "side"])
    c_smode = pick(["SideMode", "side_mode", "PortfolioSide", "portfolio_side"])
    c_stop  = pick(["StopATR", "stop_atr"])
    c_tgt   = pick(["TargetATR", "target_atr"])
    c_qty   = pick(["Qty", "qty", "quantity"])
    c_pnl   = pick(["PnL", "pnl"])
    c_entry_px = pick(["EntryPrice", "entry_price"])
    c_exit_px  = pick(["ExitPrice", "exit_price"])

    missing = [k for k, v in [("EntryDate", c_entry), ("ExitDate", c_exit), ("Ticker/Symbol", c_sym), ("Rmult", c_r)] if v is None]
    if missing:
        raise RuntimeError(f"Trade logs missing required columns: {missing}")

    t["entry_date"] = s_day(t[c_entry], fmt=None, default_dayfirst=False)
    t["exit_date"]  = s_day(t[c_exit],  fmt=None, default_dayfirst=False)

    t["symbol"]     = t[c_sym].astype(str).str.upper().str.strip()
    t["rmult"]      = pd.to_numeric(t[c_r], errors="coerce")

    t["regime"] = t[c_reg].astype(str).str.lower().str.strip() if c_reg else None
    t["side"] = t[c_side].astype(str).str.lower().str.strip() if c_side else None
    t["side_mode"] = t[c_smode].astype(str).str.lower().str.strip() if c_smode else None
    t["stop_atr"] = pd.to_numeric(t[c_stop], errors="coerce") if c_stop else np.nan
    t["target_atr"] = pd.to_numeric(t[c_tgt], errors="coerce") if c_tgt else np.nan

    t["qty"] = pd.to_numeric(t[c_qty], errors="coerce") if c_qty else np.nan
    t["entry_price"] = pd.to_numeric(t[c_entry_px], errors="coerce") if c_entry_px else np.nan
    t["exit_price"]  = pd.to_numeric(t[c_exit_px], errors="coerce") if c_exit_px else np.nan
    t["pnl"] = pd.to_numeric(t[c_pnl], errors="coerce") if c_pnl else np.nan

    need_fill = t["regime"].isna() | t["side_mode"].isna() | t["stop_atr"].isna() | t["target_atr"].isna()
    if need_fill.any():
        parsed = t.loc[need_fill, "__srcfile"].map(_parse_key_from_srcfile)
        reg, sm, st, tg = [], [], [], []
        for v in parsed.tolist():
            if v is None:
                reg.append(None); sm.append(None); st.append(np.nan); tg.append(np.nan)
            else:
                r, s_mode, s, t_ = v
                reg.append(r); sm.append(s_mode); st.append(float(s)); tg.append(float(t_))
        idx = t.loc[need_fill].index
        t.loc[idx, "regime"] = t.loc[idx, "regime"].fillna(pd.Series(reg, index=idx))
        t.loc[idx, "side_mode"] = t.loc[idx, "side_mode"].fillna(pd.Series(sm, index=idx))
        t.loc[idx, "stop_atr"] = t.loc[idx, "stop_atr"].fillna(pd.Series(st, index=idx))
        t.loc[idx, "target_atr"] = t.loc[idx, "target_atr"].fillna(pd.Series(tg, index=idx))

    need_pnl = t["pnl"].isna()
    if need_pnl.any() and t["qty"].notna().any() and t["entry_price"].notna().any() and t["exit_price"].notna().any():
        long_mask = (t["side"] == "long")
        short_mask = (t["side"] == "short")
        pnl_calc = pd.Series(np.nan, index=t.index, dtype="float64")
        pnl_calc.loc[long_mask] = (t.loc[long_mask, "exit_price"] - t.loc[long_mask, "entry_price"]) * t.loc[long_mask, "qty"]
        pnl_calc.loc[short_mask] = (t.loc[short_mask, "entry_price"] - t.loc[short_mask, "exit_price"]) * t.loc[short_mask, "qty"]
        t.loc[need_pnl, "pnl"] = pnl_calc.loc[need_pnl]

    t = t.dropna(subset=["symbol", "entry_date", "exit_date", "rmult", "regime", "side_mode", "side"]).copy()
    t["regime"] = t["regime"].astype(str).str.lower().str.strip()
    t["side_mode"] = t["side_mode"].astype(str).str.lower().str.strip()
    t["side"] = t["side"].astype(str).str.lower().str.strip()

    if EXCLUDE_ETF:
        t = t[~t["symbol"].map(_is_etf_symbol)].copy()

    assert_day_dt(t, "entry_date", name="[TRADES] ")
    assert_day_dt(t, "exit_date",  name="[TRADES] ")

    bad_r = int((t["rmult"].abs() > R_ABS_MAX_SANITY).sum())
    log.info(f"[TRADES] Loaded rows={len(t):,} symbols={t['symbol'].nunique():,} |R|>{R_ABS_MAX_SANITY} count={bad_r:,}")
    return t


# =========================================================
# EQUITY CAGR helper
# =========================================================
def equity_cagr_from_slice(eq_df: pd.DataFrame, start_ts: pd.Timestamp, end_ts: pd.Timestamp) -> Tuple[float, int]:
    if eq_df is None or eq_df.empty:
        return np.nan, 0
    sl = eq_df[(eq_df["date"] >= start_ts) & (eq_df["date"] <= end_ts)].sort_values("date")
    if sl.empty or len(sl) < 2:
        return np.nan, int(len(sl))
    e0 = float(sl["equity"].iloc[0])
    e1 = float(sl["equity"].iloc[-1])
    d0 = pd.Timestamp(sl["date"].iloc[0])
    d1 = pd.Timestamp(sl["date"].iloc[-1])
    days = max(1, int((d1 - d0).days))
    if e0 <= 0 or e1 <= 0:
        return np.nan, int(len(sl))
    cagr = (e1 / e0) ** (365.0 / days) - 1.0
    return float(cagr), int(len(sl))


# =========================================================
# OHLCV Rules: duplicate dedupe + gap segments
# =========================================================
def dedupe_duplicate_dates_closest_close(df_sym):
    if df_sym.empty:
        return df_sym
    df_sym = df_sym.sort_values(["date"]).copy()
    keep_idx = []
    prev_close = np.nan

    for d, g in df_sym.groupby("date", sort=True):
        if len(g) == 1:
            idx = g.index[0]
            row = g.iloc[0]
        else:
            gg = g[pd.notna(g["close"])].copy()
            if gg.empty:
                idx = g.index[-1]
                row = g.loc[idx]
            else:
                if pd.isna(prev_close):
                    if "volume" in gg.columns and gg["volume"].notna().any():
                        idx = gg.sort_values(["volume"], ascending=False).index[0]
                        row = gg.loc[idx]
                    else:
                        idx = gg.index[-1]
                        row = gg.loc[idx]
                else:
                    gg["absdiff"] = (gg["close"].astype(float) - float(prev_close)).abs()
                    sort_cols = ["absdiff"]
                    asc = [True]
                    if "volume" in gg.columns:
                        sort_cols.append("volume")
                        asc.append(False)
                    idx = gg.sort_values(sort_cols, ascending=asc).index[0]
                    row = gg.loc[idx]

        keep_idx.append(idx)
        if pd.notna(row["close"]):
            prev_close = float(row["close"])

    return df_sym.loc[keep_idx].copy().sort_values(["date"])

def add_gap_segments(px: pd.DataFrame) -> pd.DataFrame:
    px = px.sort_values(["symbol", "date"]).copy()
    assert_day_dt(px, "date", name="[OHLCV] ")
    px["gap_days"] = px.groupby("symbol", sort=False)["date"].diff().dt.days
    px["seg_id"] = px["gap_days"].ge(SUSP_GAP_DAYS).groupby(px["symbol"]).cumsum().fillna(0).astype(int)
    return px


# =========================================================
# Load OHLCV subset + build series_map + seg_map (cached)
# =========================================================
@dataclass
class SymSeries:
    dates64: np.ndarray
    seg_id: np.ndarray
    feats: np.ndarray

def _find_ohlcv_columns(head_cols):
    cols = {c.lower().strip(): c for c in head_cols}
    sym_col = cols.get("tradingsymbol") or cols.get("symbol")
    date_col = cols.get("date") or cols.get("timestamp") or cols.get("datetime")
    o_col = cols.get("open") or cols.get("open_price") or cols.get("openprice")
    h_col = cols.get("high") or cols.get("high_price") or cols.get("highprice")
    l_col = cols.get("low")  or cols.get("low_price") or cols.get("lowprice")
    c_col = cols.get("close") or cols.get("close_price") or cols.get("closeprice")
    v_col = cols.get("volume") or cols.get("ttl_trd_qnty") or cols.get("vol")
    return sym_col, date_col, o_col, h_col, l_col, c_col, v_col

def load_ohlcv_subset_build_series(csv_path, symbols_needed):
    cache_series, cache_seg, cache_meta = build_ohlcv_cache_paths(csv_path, symbols_needed)
    if os.path.exists(cache_series) and os.path.exists(cache_seg) and os.path.exists(cache_meta):
        try:
            meta = json.load(open(cache_meta, "r"))
            log.info(f"[OHLCV][CACHE] Loading series/seg from tag={meta.get('tag','?')} path={os.path.basename(cache_series)}")
            series_map = joblib.load(cache_series)
            seg_map = pd.read_pickle(cache_seg)
            assert_day_dt(seg_map, "date", name="[SEG_MAP] ")
            return series_map, seg_map
        except Exception as e:
            log.warning(f"[OHLCV][CACHE] load failed, rebuilding. reason={e}")

    head = pd.read_csv(csv_path, nrows=5)
    head.columns = [c.strip() for c in head.columns]
    sym_col, date_col, o_col, h_col, l_col, c_col, v_col = _find_ohlcv_columns(head.columns)
    if sym_col is None or date_col is None or c_col is None:
        raise RuntimeError("OHLCV missing required columns (symbol/date/close).")

    usecols = [sym_col, date_col, c_col]
    for x in (o_col, h_col, l_col, v_col):
        if x is not None:
            usecols.append(x)
    usecols = list(dict.fromkeys(usecols))

    symset = set([s.upper() for s in symbols_needed])

    sample = pd.read_csv(csv_path, usecols=[date_col], nrows=2500)
    _parsed_s, fmt, note = parse_date_series_with_inference(sample[date_col], default_dayfirst=False, sample_n=400)
    log.info(f"[OHLCV][DATEFMT] fmt={fmt} note={note}")

    parts = []
    t0 = time.time()
    rows_raw = 0
    rows_kept = 0
    for i, chunk in enumerate(pd.read_csv(csv_path, usecols=usecols, chunksize=700_000), 1):
        chunk.columns = [c.strip() for c in chunk.columns]
        chunk = chunk.rename(columns={sym_col: "symbol", date_col: "date_raw", c_col: "close"})
        if o_col is not None and o_col in chunk.columns: chunk = chunk.rename(columns={o_col: "open"})
        if h_col is not None and h_col in chunk.columns: chunk = chunk.rename(columns={h_col: "high"})
        if l_col is not None and l_col in chunk.columns: chunk = chunk.rename(columns={l_col: "low"})
        if v_col is not None and v_col in chunk.columns: chunk = chunk.rename(columns={v_col: "volume"})

        rows_raw += len(chunk)
        chunk["symbol"] = chunk["symbol"].astype(str).str.upper().str.strip()
        chunk = chunk[chunk["symbol"].isin(symset)].copy()
        if chunk.empty:
            continue

        if fmt is not None:
            dt = s_day(chunk["date_raw"].astype(str).map(_date_part), fmt=fmt, default_dayfirst=False)
            chunk["date"] = dt
        else:
            parsed, _, _ = parse_date_series_with_inference(chunk["date_raw"], default_dayfirst=False, sample_n=400)
            chunk["date"] = parsed

        for col in ["open", "high", "low", "close", "volume"]:
            if col in chunk.columns:
                chunk[col] = pd.to_numeric(chunk[col], errors="coerce")

        chunk = chunk.dropna(subset=["symbol", "date", "close"]).copy()
        if chunk.empty:
            continue

        for col in ["open", "high", "low", "volume"]:
            if col not in chunk.columns:
                chunk[col] = 0.0 if col == "volume" else chunk["close"]

        assert_day_dt(chunk, "date", name="[OHLCV_CHUNK] ")
        rows_kept += len(chunk)
        parts.append(chunk[["symbol", "date", "open", "high", "low", "close", "volume"]])

        if i % 8 == 0:
            log.info(f"[OHLCV] chunks={i} raw_rows={rows_raw:,} kept_rows={rows_kept:,} parts={len(parts)} elapsed={time.time()-t0:.1f}s")

    if not parts:
        raise RuntimeError("No OHLCV rows found for requested symbols.")
    px = pd.concat(parts, ignore_index=True)
    log.info(f"[OHLCV] raw subset rows={len(px):,} symbols={px['symbol'].nunique():,}")

    deduped = []
    dup_symbols = 0
    for sym, g in px.groupby("symbol", sort=False):
        g = g.sort_values("date").copy()
        if g.duplicated(subset=["date"]).any():
            dup_symbols += 1
            g = dedupe_duplicate_dates_closest_close(g)
        deduped.append(g)
    px2 = pd.concat(deduped, ignore_index=True).sort_values(["symbol", "date"])
    log.info(f"[OHLCV] after dedupe rows={len(px2):,} dup(symbol,date)={int(px2.duplicated(['symbol','date']).sum()):,} dup_syms={dup_symbols:,}")
    assert_day_dt(px2, "date", name="[OHLCV_DEDUP] ")

    px2 = add_gap_segments(px2)

    series_map: Dict[str, SymSeries] = {}
    seg_rows = []
    kept_syms = 0
    skipped_short = 0

    for sym, g in px2.groupby("symbol", sort=False):
        g = g.sort_values("date").copy()
        g = g[(g["open"] > 0) & (g["high"] > 0) & (g["low"] > 0) & (g["close"] > 0)].copy()
        if len(g) < 30:
            skipped_short += 1
            continue

        close = g["close"].astype(float).values
        open_ = g["open"].astype(float).values
        high = g["high"].astype(float).values
        low  = g["low"].astype(float).values
        vol  = g["volume"].fillna(0.0).astype(float).values

        prev_close = np.roll(close, 1); prev_close[0] = close[0]
        log_ret_close = np.log(close / np.maximum(1e-12, prev_close))
        hl_range = np.log(high / np.maximum(1e-12, low))
        oc_move  = np.log(close / np.maximum(1e-12, open_))
        vlog = np.log1p(np.maximum(0.0, vol))
        vol_chg = np.diff(vlog, prepend=vlog[0])

        feats = np.vstack([log_ret_close, hl_range, oc_move, vol_chg]).T.astype(np.float32)

        dates64 = g["date"].values.astype("datetime64[D]")
        seg_id = g["seg_id"].astype(int).values.astype(np.int32)

        series_map[sym] = SymSeries(dates64=dates64, seg_id=seg_id, feats=feats)
        seg_rows.append(pd.DataFrame({"symbol": sym, "date": g["date"].values, "seg_id": seg_id}))
        kept_syms += 1

    if not seg_rows:
        raise RuntimeError("No seg_map rows built.")
    seg_map = pd.concat(seg_rows, ignore_index=True)
    assert_day_dt(seg_map, "date", name="[SEG_MAP_BUILD] ")

    pd.DataFrame({"symbol": list(series_map.keys()), "n_bars": [len(v.dates64) for v in series_map.values()]}).to_csv(
        os.path.join(OUT_DIR, "caches", f"debug_series_symbols_{os.path.basename(cache_series).replace('.pkl','.csv')}"), index=False
    )

    seg_map.to_pickle(cache_seg)
    joblib.dump(series_map, cache_series)
    meta = {
        "tag": os.path.basename(cache_series).replace("ohlcv_series_", "").replace(".pkl", ""),
        "csv_path": csv_path,
        "csv_size": int(os.stat(csv_path).st_size),
        "csv_mtime_ns": int(os.stat(csv_path).st_mtime_ns),
        "n_symbols_requested": int(len(symset)),
        "symbol_hash": _stable_symbol_hash(symbols_needed),
        "n_symbols_kept": int(kept_syms),
        "n_symbols_skipped_short": int(skipped_short),
        "n_rows_seg_map": int(len(seg_map)),
    }
    json.dump(meta, open(cache_meta, "w"), indent=2)
    log.info(f"[OHLCV][CACHE] Saved tag={meta['tag']} symbols_kept={kept_syms:,} symbols_short={skipped_short:,} seg_rows={len(seg_map):,}")
    return series_map, seg_map

def apply_gap_skip(trades: pd.DataFrame, seg_map: pd.DataFrame) -> pd.DataFrame:
    t = trades.copy()
    assert_day_dt(t, "entry_date", name="[GAP/TRADES] ")
    assert_day_dt(t, "exit_date",  name="[GAP/TRADES] ")
    assert_day_dt(seg_map, "date", name="[GAP/SEG] ")

    t = t.merge(seg_map.rename(columns={"date": "entry_date", "seg_id": "entry_seg"}),
                on=["symbol", "entry_date"], how="left")
    t = t.merge(seg_map.rename(columns={"date": "exit_date", "seg_id": "exit_seg"}),
                on=["symbol", "exit_date"], how="left")

    t["gap_skip"] = 0
    t.loc[t["entry_seg"].isna() | t["exit_seg"].isna(), "gap_skip"] = 1
    t.loc[(t["entry_seg"].notna()) & (t["exit_seg"].notna()) & (t["entry_seg"] != t["exit_seg"]), "gap_skip"] = 1

    log.info(f"[GAP] missing seg_id: entry={int(t['entry_seg'].isna().sum()):,} exit={int(t['exit_seg'].isna().sum()):,}")
    log.info(f"[GAP] gap_skip trades={int(t['gap_skip'].sum()):,} / {len(t):,}")
    return t


# =========================================================
# Static features (CS z/rank) no apply
# =========================================================
def add_cs_z_rank_inplace(df: pd.DataFrame, cols: List[str], group_col: str) -> List[str]:
    X = df[cols].apply(pd.to_numeric, errors="coerce")
    g = df[group_col]
    mean = X.groupby(g, sort=False).transform("mean")
    std = X.groupby(g, sort=False).transform(lambda s: s.std(ddof=0))
    z = (X - mean) / std
    z = z.replace([np.inf, -np.inf], np.nan)
    rk = X.groupby(g, sort=False).rank(pct=True, method="average")
    out_cols = []
    for c in cols:
        zc = f"{c}_cs_z"
        rc = f"{c}_cs_rank"
        df[zc] = z[c].astype(np.float32)
        df[rc] = rk[c].astype(np.float32)
        out_cols += [zc, rc]
    return out_cols


# =========================================================
# ROCKET (deterministic kernels)
# =========================================================
@dataclass
class RocketKernel:
    channel: int
    length: int
    dilation: int
    weights: np.ndarray
    bias: np.float32
    idx_cols: List[np.ndarray]

class RocketFeaturizer:
    def __init__(self, window, n_channels, n_kernels, max_dilation, seed):
        self.window = int(window)
        self.n_channels = int(n_channels)
        self.n_kernels = int(n_kernels)
        self.max_dilation = int(max_dilation)
        self.seed = int(seed)
        self.kernels: List[RocketKernel] = []

    def _sample_dilation(self, rng, L, k_len):
        candidates = []
        d = 1
        while d <= self.max_dilation:
            if (k_len - 1) * d < L:
                candidates.append(d)
            d *= 2
        return int(rng.choice(candidates)) if candidates else 1

    def _sample_length(self, rng, L):
        max_len = min(11, L)
        lens = [2, 3, 5, 7, 9, 11]
        lens = [k for k in lens if k <= max_len]
        return int(rng.choice(lens)) if lens else min(2, L)

    def build(self):
        rng = np.random.RandomState(self.seed)
        L = self.window
        C = self.n_channels
        tries = 0
        while len(self.kernels) < self.n_kernels and tries < self.n_kernels * 50:
            tries += 1
            ch = int(rng.randint(0, C))
            k_len = self._sample_length(rng, L)
            d = self._sample_dilation(rng, L, k_len)
            pos_count = L - (k_len - 1) * d
            if pos_count <= 0:
                continue
            w = rng.normal(0, 1, size=k_len).astype(np.float32)
            w = (w - w.mean()).astype(np.float32)
            b = np.float32(rng.normal(0, 1))
            pos = np.arange(pos_count, dtype=np.int32)
            idx_cols = [pos + (j * d) for j in range(k_len)]
            self.kernels.append(RocketKernel(ch, k_len, d, w, b, idx_cols))
        return self

    def transform_batch(self, X):
        X = X.astype(np.float32, copy=False)
        n, L, _ = X.shape
        out = np.empty((n, 2 * len(self.kernels)), dtype=np.float32)
        for ki, k in enumerate(self.kernels):
            x = X[:, :, k.channel]
            y = np.full((n, k.idx_cols[0].shape[0]), k.bias, dtype=np.float32)
            for j, idx in enumerate(k.idx_cols):
                y += k.weights[j] * x[:, idx]
            out[:, 2 * ki] = y.max(axis=1)
            out[:, 2 * ki + 1] = (y > 0).mean(axis=1).astype(np.float32)
        return out

def build_featurizers(rocket_cfg: dict, seed: int) -> Dict[int, RocketFeaturizer]:
    cfg = normalize_rocket_cfg(rocket_cfg)
    C = len(ROCKET_CHANNELS)
    fe = {}
    for L in ROCKET_WINDOWS:
        nk = int(cfg["kernels"][L])
        fe[L] = RocketFeaturizer(L, C, nk, cfg["max_dilation"], seed + 1000 * L).build()
    return fe

def rocket_feature_dim(rocket_cfg: dict) -> int:
    cfg = normalize_rocket_cfg(rocket_cfg)
    return sum(2 * int(cfg["kernels"][L]) for L in ROCKET_WINDOWS)

def build_windows_for_batch_rows(symbols: np.ndarray,
                                entry_dates64: np.ndarray,
                                series_map: Dict[str, SymSeries],
                                L: int) -> Tuple[np.ndarray, np.ndarray]:
    n = len(symbols)
    C = len(ROCKET_CHANNELS)
    X = np.zeros((n, L, C), dtype=np.float32)
    ok = np.zeros((n,), dtype=np.uint8)

    uniq_syms = pd.unique(symbols)
    ar = np.arange(L, dtype=np.int64)[None, :]

    for sym in uniq_syms:
        s = series_map.get(sym)
        if s is None:
            continue
        pos = np.where(symbols == sym)[0]
        if pos.size == 0:
            continue

        dates64 = s.dates64
        seg = s.seg_id
        feats = s.feats

        ent = entry_dates64[pos]
        p = np.searchsorted(dates64, ent, side="left") - 1
        valid = p >= (L - 1)
        if not np.any(valid):
            continue

        pos_v = pos[valid]
        p_v = p[valid].astype(np.int64)
        start = p_v - (L - 1)

        seg_ok = (seg[p_v] == seg[start])
        if not np.any(seg_ok):
            continue

        pos_ok = pos_v[seg_ok]
        start_ok = start[seg_ok]
        take_idx = start_ok[:, None] + ar
        X[pos_ok, :, :] = feats[take_idx, :]
        ok[pos_ok] = 1

    return X, ok

def write_rocket_memmap(data: pd.DataFrame,
                        series_map: Dict[str, SymSeries],
                        rocket_cfg: dict,
                        seed: int,
                        out_path: str,
                        batch_size: int) -> Tuple[str, np.ndarray]:
    cfg = normalize_rocket_cfg(rocket_cfg)
    n = len(data)
    d = rocket_feature_dim(cfg)
    mm = np.memmap(out_path, dtype=FEATURE_DTYPE, mode="w+", shape=(n, d))

    featurizers = build_featurizers(cfg, seed)

    offsets = {}
    off = 0
    for L in ROCKET_WINDOWS:
        span = 2 * int(cfg["kernels"][L])
        offsets[L] = (off, off + span)
        off += span

    ok_all = np.ones((n,), dtype=np.uint8)
    symbols_all = data["symbol"].values.astype(str)
    entry_dates64_all = data["entry_date"].values.astype("datetime64[D]")

    t0 = time.time()
    for start in range(0, n, batch_size):
        end = min(n, start + batch_size)
        sy = symbols_all[start:end]
        ed = entry_dates64_all[start:end]

        ok_windows = []
        for L in ROCKET_WINDOWS:
            Xw, okw = build_windows_for_batch_rows(sy, ed, series_map, L=L)
            Fw = featurizers[L].transform_batch(Xw)
            a, b = offsets[L]
            mm[start:end, a:b] = Fw.astype(FEATURE_DTYPE)
            ok_windows.append(okw)

        okb = ok_windows[0].copy()
        for k in ok_windows[1:]:
            okb = (okb & k).astype(np.uint8)
        ok_all[start:end] = okb

        if (start // batch_size) % 10 == 0:
            log.info(f"[ROCKET/MMAP] rows {start:,}-{end:,}/{n:,} elapsed={time.time()-t0:.1f}s")

    mm.flush()
    return out_path, ok_all

def open_memmap(path: str, shape: Tuple[int, int]) -> np.memmap:
    return np.memmap(path, dtype=FEATURE_DTYPE, mode="r", shape=shape)

def get_X_rows(idx: np.ndarray, mm_rocket: np.memmap, mm_static: Optional[np.memmap]) -> np.ndarray:
    Xr = np.asarray(mm_rocket[idx, :], dtype=TRAIN_DTYPE)
    if mm_static is None:
        return Xr
    Xs = np.asarray(mm_static[idx, :], dtype=TRAIN_DTYPE)
    return np.hstack([Xr, Xs]).astype(TRAIN_DTYPE, copy=False)


# =========================================================
# Platt calibration
# =========================================================
@dataclass
class PlattCalibrator:
    a: float
    b: float
    def predict_proba(self, scores: np.ndarray) -> np.ndarray:
        z = np.clip(self.a * scores + self.b, -50, 50)
        return (1.0 / (1.0 + np.exp(-z))).astype(np.float32)

def fit_platt(scores: np.ndarray, y: np.ndarray) -> Optional[PlattCalibrator]:
    y = np.asarray(y).astype(int)
    if len(np.unique(y)) < 2:
        return None
    lr = LogisticRegression(solver="lbfgs", max_iter=2000)
    lr.fit(scores.reshape(-1, 1), y)
    return PlattCalibrator(a=float(lr.coef_.ravel()[0]), b=float(lr.intercept_.ravel()[0]))

def sigmoid(x: np.ndarray) -> np.ndarray:
    z = np.clip(x, -50, 50)
    return (1.0 / (1.0 + np.exp(-z))).astype(np.float32)


# =========================================================
# Rolling folds + winners (NOLEAK)
# =========================================================
def build_yearly_folds(eval_start: pd.Timestamp, eval_end: pd.Timestamp, lookback_years: int, first_test_year: Optional[int]) -> List[dict]:
    if first_test_year is None:
        first_test_year = int(eval_start.year) + 1
    folds = []
    for y in range(first_test_year, int(eval_end.year) + 1):
        train_end = pd.Timestamp(y - 1, 12, 31).normalize()
        test_start = pd.Timestamp(y, 1, 1).normalize()
        test_end = min(pd.Timestamp(y, 12, 31).normalize(), eval_end)

        train_start = pd.Timestamp(max(int(eval_start.year), y - lookback_years), 1, 1).normalize()
        if train_start < eval_start:
            train_start = eval_start

        if test_end < test_start:
            continue
        if int((test_end - test_start).days) + 1 < ROLLING_MIN_TEST_DAYS:
            continue
        if train_end < train_start:
            continue

        folds.append(dict(
            fold_year=y,
            train_start=train_start,
            train_end=train_end,
            test_start=test_start,
            test_end=test_end,
        ))
    return folds

def select_winners(eq_map: Dict[Tuple[str, str, float, float], pd.DataFrame],
                   data: pd.DataFrame,
                   train_start: pd.Timestamp,
                   train_end: pd.Timestamp) -> Tuple[pd.DataFrame, pd.DataFrame]:
    dwin = data[
        (data["entry_date"] >= train_start) &
        (data["entry_date"] <= train_end) &
        (data["exit_date"]  <= train_end)
    ]
    counts = dwin.groupby(["regime", "side_mode", "stop_r", "tgt_r"], dropna=False)["row_id"].size().rename("n_trades").reset_index()

    rows = []
    for (regime, side_mode, stop, tgt), eq in eq_map.items():
        stop_r = round(float(stop), 6)
        tgt_r = round(float(tgt), 6)
        cagr, n_eq = equity_cagr_from_slice(eq, train_start, train_end)
        rows.append(dict(
            regime=regime, side_mode=side_mode, stop_r=stop_r, tgt_r=tgt_r,
            baseline_cagr=cagr, equity_points=n_eq
        ))

    tbl = pd.DataFrame(rows)
    tbl = tbl.merge(counts, on=["regime", "side_mode", "stop_r", "tgt_r"], how="left")
    tbl["n_trades"] = tbl["n_trades"].fillna(0).astype(int)

    ok = (
        (tbl["equity_points"] >= int(WINNER_MIN_EQUITY_DAYS)) &
        (tbl["baseline_cagr"].notna()) &
        (tbl["baseline_cagr"] > float(WINNER_CAGR_MIN)) &
        (tbl["n_trades"] >= int(WINNER_MIN_TRADES))
    )
    tbl["is_winner"] = ok.astype(int)
    winners = tbl[tbl["is_winner"] == 1].copy()
    return tbl, winners


# =========================================================
# Magnitude BINS
# =========================================================
def _make_bins_from_quantiles(x: np.ndarray, qs: List[float]) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if x.size < 200:
        edges = [0.0, np.quantile(x, 0.7) if x.size else 1.0, np.quantile(x, 0.9) if x.size else 2.0, np.inf]
        return np.array(edges, dtype=float)
    cuts = [float(np.quantile(x, q)) for q in qs]
    cuts = [c for c in cuts if c > 0 and np.isfinite(c)]
    edges = [0.0] + sorted(list(dict.fromkeys(cuts))) + [np.inf]
    edges2 = [edges[0]]
    for v in edges[1:]:
        if v > edges2[-1] * 1.0000001:
            edges2.append(v)
    if len(edges2) < 4:
        m = float(np.median(x)) if x.size else 1.0
        edges2 = [0.0, m, m * 1.5 + 1e-6, np.inf]
    return np.array(edges2, dtype=float)

def _bin_indices(x: np.ndarray, edges: np.ndarray) -> np.ndarray:
    x = np.maximum(0.0, np.asarray(x, dtype=float))
    k = len(edges) - 1
    idx = np.searchsorted(edges, x, side="right") - 1
    return np.clip(idx, 0, k - 1).astype(int)

def _bin_representatives(x: np.ndarray, ybin: np.ndarray, k: int) -> np.ndarray:
    reps = np.zeros((k,), dtype=np.float32)
    for i in range(k):
        vals = x[ybin == i]
        vals = vals[np.isfinite(vals)]
        if vals.size == 0:
            reps[i] = 0.0
        elif vals.size < 50:
            reps[i] = float(np.median(vals))
        else:
            reps[i] = float(np.mean(vals))
    return reps.astype(np.float32)

@dataclass
class Stage2MagModel:
    edges: np.ndarray
    reps: np.ndarray
    clf: RandomForestClassifier

def train_mag_bin_model(x_mag: np.ndarray, X: np.ndarray) -> Optional[Stage2MagModel]:
    x_mag = np.asarray(x_mag, dtype=float)
    x_mag = x_mag[np.isfinite(x_mag)]
    if x_mag.size < 200:
        return None
    edges = _make_bins_from_quantiles(x_mag, MAG_BIN_QUANTILES)
    ybin = _bin_indices(x_mag, edges)
    clf = RandomForestClassifier(**MAG_RF_PARAMS)
    clf.fit(X, ybin)
    reps = _bin_representatives(x_mag, ybin, k=(len(edges) - 1))
    return Stage2MagModel(edges=edges, reps=reps, clf=clf)

def predict_mag_expectation(model: Optional[Stage2MagModel], X: np.ndarray) -> np.ndarray:
    if model is None:
        return np.zeros((X.shape[0],), dtype=np.float32)
    proba = model.clf.predict_proba(X).astype(np.float32)
    reps = model.reps.reshape(1, -1).astype(np.float32)
    return (proba * reps).sum(axis=1).astype(np.float32)


# =========================================================
# Stage1 (p_win)
# =========================================================
@dataclass
class Stage1Model:
    ridge: RidgeClassifier
    cal: Optional[PlattCalibrator]
    rf: RandomForestClassifier

def train_stage1(X_tr: np.ndarray, y_tr: np.ndarray, X_va: np.ndarray, y_va: np.ndarray, sample_weight: np.ndarray) -> Stage1Model:
    ridge = RidgeClassifier(alpha=RIDGE_ALPHA, random_state=42)
    ridge.fit(X_tr, y_tr, sample_weight=sample_weight)
    s_va = ridge.decision_function(X_va).astype(np.float32)
    cal = fit_platt(s_va, y_va)

    rf = RandomForestClassifier(**RF_PWIN_PARAMS)
    rf.fit(X_tr, y_tr, sample_weight=sample_weight)
    return Stage1Model(ridge=ridge, cal=cal, rf=rf)

def predict_pwin(stage1: Stage1Model, X: np.ndarray) -> np.ndarray:
    s = stage1.ridge.decision_function(X).astype(np.float32)
    p_r = stage1.cal.predict_proba(s) if stage1.cal is not None else sigmoid(s)
    p_f = stage1.rf.predict_proba(X)[:, 1].astype(np.float32)
    p = (W_PWIN_RIDGE * p_r + W_PWIN_RF * p_f).astype(np.float32)
    return np.clip(p, 1e-5, 1.0 - 1e-5)


# =========================================================
# Fold model training per bucket (regime x side)
# =========================================================
@dataclass
class BucketModels:
    stage1: Stage1Model
    win_mag: Optional[Stage2MagModel]
    loss_mag: Optional[Stage2MagModel]

def _choose_val_dates_noleak(sub: pd.DataFrame) -> Tuple[Optional[set], Optional[pd.Timestamp], Optional[pd.DataFrame], Optional[pd.DataFrame], str]:
    dates = np.array(sorted(sub["entry_date"].unique()))
    if dates.size == 0:
        return None, None, None, None, "no_dates"

    val_dates = pick_val_dates(dates)
    if not val_dates:
        return None, None, None, None, "no_val_dates"

    vd = len(val_dates)
    vd = max(VALID_DAYS_MIN, min(vd, dates.size))
    step = 5

    best = None
    while vd >= VALID_DAYS_MIN:
        tail = set(dates[-vd:])
        val_min = pd.Timestamp(dates[-vd]).normalize()
        tr_base = sub[sub["exit_date"] < val_min].copy()
        va = sub[sub["entry_date"].isin(tail)].copy()

        if len(tr_base) >= MIN_TRAIN_ROWS and len(va) >= MIN_VAL_ROWS and tr_base["y_win"].nunique() >= 2 and va["y_win"].nunique() >= 2:
            best = (tail, val_min, tr_base, va, f"ok_vd={vd}")
            break

        vd2 = vd - step
        if vd2 < VALID_DAYS_MIN:
            vd2 = VALID_DAYS_MIN - 1
        vd = vd2

    if best is None:
        tail = set(dates[-max(VALID_DAYS_MIN, min(len(dates), len(val_dates))):])
        val_min = pd.Timestamp(sorted(list(tail))[0]).normalize()
        tr_base = sub[sub["exit_date"] < val_min].copy()
        va = sub[sub["entry_date"].isin(tail)].copy()
        return None, None, tr_base, va, "no_feasible_split"
    return best

def train_fold_bucket_models(fold_tag: str,
                             fold_train: pd.DataFrame,
                             mm_rocket: np.memmap,
                             mm_static: Optional[np.memmap],
                             out_models_dir: str) -> Tuple[Dict[Tuple[str,str], BucketModels], pd.DataFrame]:
    os.makedirs(out_models_dir, exist_ok=True)
    stats = []
    models: Dict[Tuple[str,str], BucketModels] = {}

    for regime in ["bull", "bear"]:
        for side in ["long", "short"]:
            gkey = (regime, side)
            sub = fold_train[(fold_train["regime"] == regime) & (fold_train["side"] == side)].copy()
            if sub.empty:
                stats.append(dict(fold=fold_tag, regime=regime, side=side, status="no_rows"))
                continue

            ok_rate = float((sub["rocket_ok_all"] == 1).mean()) if "rocket_ok_all" in sub.columns else 1.0
            sub_ok = sub[sub["rocket_ok_all"] == 1].copy() if "rocket_ok_all" in sub.columns else sub
            if "rocket_ok_all" in sub.columns and len(sub_ok) >= max(MIN_TRAIN_ROWS, int(0.6 * len(sub))):
                sub = sub_ok

            val_dates, val_min, tr_base, va, note = _choose_val_dates_noleak(sub)
            if val_dates is None or tr_base is None or va is None or val_min is None:
                stats.append(dict(
                    fold=fold_tag, regime=regime, side=side, status="too_small_or_single_class",
                    n_total=int(len(sub)), ok_rate=float(ok_rate),
                    n_train=int(len(tr_base)) if tr_base is not None else 0,
                    n_val=int(len(va)) if va is not None else 0,
                    note=str(note)
                ))
                continue

            tr_base = subsample_stratified(tr_base, "y_win", MAX_TRAIN_SAMPLES_PER_BUCKET, seed=101)
            va = subsample_stratified(va, "y_win", MAX_VAL_SAMPLES_PER_BUCKET, seed=202)

            idx_tr = tr_base["row_id"].values.astype(np.int64)
            idx_va = va["row_id"].values.astype(np.int64)

            X_tr = get_X_rows(idx_tr, mm_rocket, mm_static)
            X_va = get_X_rows(idx_va, mm_rocket, mm_static)
            y_tr = tr_base["y_win"].astype(int).values
            y_va = va["y_win"].astype(int).values

            pos = int((y_tr == 1).sum())
            neg = int((y_tr == 0).sum())
            w_pos = 0.5 / max(1, pos)
            w_neg = 0.5 / max(1, neg)
            sw = np.where(y_tr == 1, w_pos, w_neg).astype(np.float32)

            stage1 = train_stage1(X_tr, y_tr, X_va, y_va, sample_weight=sw)

            r_tr = pd.to_numeric(tr_base["rmult"], errors="coerce").values.astype(float)
            r_tr = np.clip(r_tr, -R_ABS_MAX_SANITY, R_ABS_MAX_SANITY)
            win_mask = (r_tr > 0)
            loss_mask = (r_tr < 0)

            win_mag = train_mag_bin_model(r_tr[win_mask], X_tr[win_mask]) if win_mask.sum() >= 400 else None
            loss_mag = train_mag_bin_model(np.abs(r_tr[loss_mask]), X_tr[loss_mask]) if loss_mask.sum() >= 400 else None

            pwin_va = predict_pwin(stage1, X_va)
            Ew_va = predict_mag_expectation(win_mag, X_va)
            El_va = predict_mag_expectation(loss_mag, X_va)
            expr_va = (pwin_va * Ew_va) - ((1.0 - pwin_va) * El_va)

            pr = float(average_precision_score(y_va, pwin_va)) if len(np.unique(y_va)) > 1 else np.nan
            roc = float(roc_auc_score(y_va, pwin_va)) if len(np.unique(y_va)) > 1 else np.nan
            r_va = np.clip(pd.to_numeric(va["rmult"], errors="coerce").fillna(0.0).values.astype(float), -R_ABS_MAX_SANITY, R_ABS_MAX_SANITY)
            ic = np.corrcoef(expr_va.astype(float), r_va.astype(float))[0, 1] if len(expr_va) > 5 else np.nan

            models[gkey] = BucketModels(stage1=stage1, win_mag=win_mag, loss_mag=loss_mag)

            joblib.dump(stage1, os.path.join(out_models_dir, f"stage1_{regime}_{side}.pkl"))
            joblib.dump(win_mag, os.path.join(out_models_dir, f"winmag_{regime}_{side}.pkl"))
            joblib.dump(loss_mag, os.path.join(out_models_dir, f"lossmag_{regime}_{side}.pkl"))

            stats.append(dict(
                fold=fold_tag, regime=regime, side=side, status="ok",
                n_total=int(len(sub)), ok_rate=float(ok_rate),
                n_train=int(len(tr_base)), n_val=int(len(va)),
                pr_auc=float(pr) if pd.notna(pr) else np.nan,
                roc_auc=float(roc) if pd.notna(roc) else np.nan,
                expr_ic=float(ic) if pd.notna(ic) else np.nan,
                win_mag_bins=int(len(win_mag.edges) - 1) if win_mag is not None else 0,
                loss_mag_bins=int(len(loss_mag.edges) - 1) if loss_mag is not None else 0,
                val_note=str(note),
            ))

            log.info(f"[FOLD {fold_tag}][MODEL {gkey}] {note} ok_rate={ok_rate:.3f} PR-AUC={pr:.4f} ROC-AUC={roc:.4f} ExpR-IC={ic if pd.notna(ic) else np.nan}")

    return models, pd.DataFrame(stats)


# =========================================================
# RR Policy (learned RR choice per stop_r,tgt_r)
# =========================================================
@dataclass
class RRPolicyPack:
    model: Ridge
    mean_r: float
    n_train: int

def train_rr_policy_models(
    fold_tag: str,
    fold_train: pd.DataFrame,
    mm_rocket: np.memmap,
    mm_static: Optional[np.memmap],
    out_models_dir: str
) -> Tuple[Dict[Tuple[str,str,float,float], RRPolicyPack], Dict[Tuple[float,float], RRPolicyPack], pd.DataFrame]:
    os.makedirs(out_models_dir, exist_ok=True)
    stats = []
    bucket_rr: Dict[Tuple[str,str,float,float], RRPolicyPack] = {}
    global_rr: Dict[Tuple[float,float], RRPolicyPack] = {}

    if not RR_POLICY_ENABLE:
        return bucket_rr, global_rr, pd.DataFrame(stats)

    sub = fold_train.copy()
    if "rocket_ok_all" in sub.columns:
        ok_rate = float((sub["rocket_ok_all"] == 1).mean())
        sub_ok = sub[sub["rocket_ok_all"] == 1].copy()
        if len(sub_ok) >= max(MIN_TRAIN_ROWS, int(0.6 * len(sub))):
            sub = sub_ok
        log.info(f"[FOLD {fold_tag}][RR] rocket_ok_rate={ok_rate:.3f} using_rows={len(sub):,}/{len(fold_train):,}")

    sub["stop_r"] = pd.to_numeric(sub["stop_r"], errors="coerce").round(6)
    sub["tgt_r"] = pd.to_numeric(sub["tgt_r"], errors="coerce").round(6)

    r = pd.to_numeric(sub["rmult"], errors="coerce").fillna(0.0).values.astype(np.float32)
    r = np.clip(r, -RR_POLICY_PRED_CLIP, RR_POLICY_PRED_CLIP).astype(np.float32)
    sub["_r_target"] = r

    if RR_POLICY_BUCKET_AWARE:
        grp_cols = ["regime", "side", "stop_r", "tgt_r"]
        for key, g in sub.groupby(grp_cols, sort=False):
            regime, side, stop_r, tgt_r = key
            n = len(g)
            if n < RR_POLICY_MIN_ROWS_PER_PAIR:
                stats.append(dict(fold=fold_tag, level="bucket", regime=regime, side=side, stop_r=float(stop_r), tgt_r=float(tgt_r),
                                  status="too_small", n_train=int(n)))
                continue

            g2 = subsample_random(g, RR_POLICY_MAX_ROWS_PER_PAIR, seed=777)
            idx = g2["row_id"].values.astype(np.int64)
            X = get_X_rows(idx, mm_rocket, mm_static)
            y = g2["_r_target"].values.astype(np.float32)

            mean_r = float(np.mean(y)) if len(y) else 0.0
            mdl = Ridge(alpha=float(RR_POLICY_RIDGE_ALPHA), fit_intercept=True, solver=str(RR_POLICY_RIDGE_SOLVER))
            mdl.fit(X, y)

            pack = RRPolicyPack(model=mdl, mean_r=mean_r, n_train=int(len(g2)))
            bucket_rr[(str(regime), str(side), float(stop_r), float(tgt_r))] = pack
            joblib.dump(pack, os.path.join(out_models_dir, f"rrpolicy_bucket_{regime}_{side}_s{stop_r}_t{tgt_r}.pkl"))

            stats.append(dict(fold=fold_tag, level="bucket", regime=regime, side=side, stop_r=float(stop_r), tgt_r=float(tgt_r),
                              status="ok", n_train=int(len(g2)), mean_r=float(mean_r)))

    if RR_POLICY_GLOBAL_FALLBACK:
        grp_cols2 = ["stop_r", "tgt_r"]
        for key, g in sub.groupby(grp_cols2, sort=False):
            stop_r, tgt_r = key
            n = len(g)
            if n < RR_POLICY_MIN_ROWS_PER_PAIR:
                stats.append(dict(fold=fold_tag, level="global", stop_r=float(stop_r), tgt_r=float(tgt_r),
                                  status="too_small", n_train=int(n)))
                continue

            g2 = subsample_random(g, RR_POLICY_MAX_ROWS_PER_PAIR, seed=778)
            idx = g2["row_id"].values.astype(np.int64)
            X = get_X_rows(idx, mm_rocket, mm_static)
            y = g2["_r_target"].values.astype(np.float32)

            mean_r = float(np.mean(y)) if len(y) else 0.0
            mdl = Ridge(alpha=float(RR_POLICY_RIDGE_ALPHA), fit_intercept=True, solver=str(RR_POLICY_RIDGE_SOLVER))
            mdl.fit(X, y)

            pack = RRPolicyPack(model=mdl, mean_r=mean_r, n_train=int(len(g2)))
            global_rr[(float(stop_r), float(tgt_r))] = pack
            joblib.dump(pack, os.path.join(out_models_dir, f"rrpolicy_global_s{stop_r}_t{tgt_r}.pkl"))

            stats.append(dict(fold=fold_tag, level="global", stop_r=float(stop_r), tgt_r=float(tgt_r),
                              status="ok", n_train=int(len(g2)), mean_r=float(mean_r)))

    return bucket_rr, global_rr, pd.DataFrame(stats)

def predict_rr_policy_for_df(
    df: pd.DataFrame,
    bucket_rr: Dict[Tuple[str,str,float,float], RRPolicyPack],
    global_rr: Dict[Tuple[float,float], RRPolicyPack],
    mm_rocket: np.memmap,
    mm_static: Optional[np.memmap],
) -> np.ndarray:
    n = len(df)
    out = np.full((n,), np.nan, dtype=np.float32)
    if n == 0 or (not RR_POLICY_ENABLE):
        return out

    idx_all = df["row_id"].values.astype(np.int64)
    regimes = df["regime"].astype(str).values
    sides = df["side"].astype(str).values
    stop_r = pd.to_numeric(df["stop_r"], errors="coerce").fillna(np.nan).values.astype(float)
    tgt_r  = pd.to_numeric(df["tgt_r"], errors="coerce").fillna(np.nan).values.astype(float)

    for start in range(0, n, PRED_BATCH):
        end = min(n, start + PRED_BATCH)
        idx = idx_all[start:end]
        X = get_X_rows(idx, mm_rocket, mm_static)

        reg = regimes[start:end]
        sd  = sides[start:end]
        st  = stop_r[start:end]
        tg  = tgt_r[start:end]

        pred = np.full((end - start,), np.nan, dtype=np.float32)
        keys = list(zip(reg, sd, st, tg))
        uniq = {}
        for i, k in enumerate(keys):
            uniq.setdefault(k, []).append(i)

        for k, pos in uniq.items():
            regime, side, s_r, t_r = k
            if not np.isfinite(s_r) or not np.isfinite(t_r):
                continue
            pack = None
            if RR_POLICY_BUCKET_AWARE:
                pack = bucket_rr.get((str(regime), str(side), float(round(s_r, 6)), float(round(t_r, 6))))
            if pack is None and RR_POLICY_GLOBAL_FALLBACK:
                pack = global_rr.get((float(round(s_r, 6)), float(round(t_r, 6))))
            if pack is None:
                continue
            pos = np.array(pos, dtype=np.int64)
            Xp = X[pos]
            yhat = pack.model.predict(Xp).astype(np.float32)
            yhat = np.clip(yhat, -RR_POLICY_PRED_CLIP, RR_POLICY_PRED_CLIP).astype(np.float32)
            pred[pos] = yhat

        out[start:end] = pred

    return out


# =========================================================
# ExpR scoring (missing-bucket fallback)
# =========================================================
def score_expr_for_df(df: pd.DataFrame,
                      bucket_models: Dict[Tuple[str,str], BucketModels],
                      mm_rocket: np.memmap,
                      mm_static: Optional[np.memmap]) -> Tuple[np.ndarray, np.ndarray]:
    n = len(df)
    out = np.full((n,), SENTINEL_SCORE, dtype=np.float32)
    valid = np.zeros((n,), dtype=np.uint8)
    if n == 0:
        return out, valid

    idx_all = df["row_id"].values.astype(np.int64)
    regimes = df["regime"].values.astype(str)
    sides = df["side"].values.astype(str)

    for start in range(0, n, PRED_BATCH):
        end = min(n, start + PRED_BATCH)
        idx = idx_all[start:end]
        X = get_X_rows(idx, mm_rocket, mm_static)

        r_chunk = regimes[start:end]
        s_chunk = sides[start:end]
        expr = np.full((end - start,), SENTINEL_SCORE, dtype=np.float32)
        v = np.zeros((end - start,), dtype=np.uint8)

        for regime in ["bull","bear"]:
            for side in ["long","short"]:
                mask = (r_chunk == regime) & (s_chunk == side)
                if not np.any(mask):
                    continue
                pack = bucket_models.get((regime, side))
                if pack is None:
                    continue
                Xp = X[mask]
                pwin = predict_pwin(pack.stage1, Xp)
                Ew = predict_mag_expectation(pack.win_mag, Xp)
                El = predict_mag_expectation(pack.loss_mag, Xp)
                expr[mask] = (pwin * Ew) - ((1.0 - pwin) * El)
                v[mask] = 1

        out[start:end] = expr
        valid[start:end] = v

    return out, valid


# =========================================================
# NEW: Rolling wR THROTTLE gate
# =========================================================
PFKey = Tuple[str, str, str, float, float]  # (regime, side, side_mode, stop_r, tgt_r)

@dataclass
class GateEvent:
    date: str
    key: str
    prev_mult: float
    new_mult: float
    wr: float
    n_trades: int
    n_days: int
    kill_streak: int
    revive_streak: int
    reason: str

@dataclass
class GateState:
    mult: float
    kill_streak: int
    revive_streak: int
    dq: deque  # of (exit_day_ts, r_used, w)
    day_counts: Counter
    sum_w: float
    sum_wr: float
    last_wr: float
    last_eval_cutoff: Optional[pd.Timestamp]

class RollingWRThrottleGate:
    def __init__(self, name: str):
        self.name = str(name)
        self.window_days = int(WR_GATE_WINDOW_DAYS)
        self.min_trades = int(WR_GATE_MIN_TRADES)
        self.min_days = int(WR_GATE_MIN_DAYS)
        self.kill_th = float(WR_GATE_KILL_TH)
        self.revive_th = float(WR_GATE_REVIVE_TH)
        self.kill_streak_req = int(WR_GATE_KILL_STREAK)
        self.revive_streak_req = int(WR_GATE_REVIVE_STREAK)
        self.throttle_mult = float(WR_GATE_THROTTLE_MULT)
        self.use_risk_weight = bool(WR_GATE_USE_RISK_WEIGHT)

        self.states: Dict[PFKey, GateState] = {}
        self.daily_rows: List[dict] = []
        self.events: List[GateEvent] = []
        self._exits_seen = 0

    @staticmethod
    def _key_str(k: PFKey) -> str:
        r, s, sm, st, tg = k
        return f"{r}|{s}|{sm}|s{st:g}|t{tg:g}"

    def get_mult(self, key: PFKey) -> float:
        st = self.states.get(key)
        return float(st.mult) if st is not None else 1.0

    def add_exit(self, key: PFKey, exit_day: pd.Timestamp, r_used: float, risk_amt: float):
        exit_day = pd.Timestamp(exit_day).normalize()
        w = float(risk_amt) if self.use_risk_weight else 1.0
        r = float(r_used)

        st = self.states.get(key)
        if st is None:
            st = GateState(
                mult=1.0,
                kill_streak=0,
                revive_streak=0,
                dq=deque(),
                day_counts=Counter(),
                sum_w=0.0,
                sum_wr=0.0,
                last_wr=np.nan,
                last_eval_cutoff=None,
            )
            self.states[key] = st

        st.dq.append((exit_day, r, w))
        st.day_counts[exit_day] += 1
        st.sum_w += w
        st.sum_wr += (w * r)
        self._exits_seen += 1

    def _prune_to_cutoff(self, st: GateState, cutoff: pd.Timestamp):
        cutoff = pd.Timestamp(cutoff).normalize()
        lo = cutoff - pd.Timedelta(days=self.window_days - 1)

        while st.dq and st.dq[0][0] < lo:
            d0, r0, w0 = st.dq.popleft()
            st.sum_w -= float(w0)
            st.sum_wr -= float(w0) * float(r0)
            st.day_counts[d0] -= 1
            if st.day_counts[d0] <= 0:
                del st.day_counts[d0]

    def update_states_for_cutoff(self, trade_day: pd.Timestamp):
        """
        Called at start of trade_day before entries.
        Uses cutoff = trade_day - 1 calendar day (performance known up to prior close).
        """
        trade_day = pd.Timestamp(trade_day).normalize()
        cutoff = (trade_day - pd.Timedelta(days=1)).normalize()

        n_keys = len(self.states)
        eval_wrs = []
        n_eval = 0
        n_throttled = 0
        n_events = 0

        for key, st in self.states.items():
            self._prune_to_cutoff(st, cutoff)

            n_tr = len(st.dq)
            n_d = len(st.day_counts)

            if n_tr < self.min_trades or n_d < self.min_days or st.sum_w <= 0:
                # no evaluation today; keep streaks unchanged
                continue

            wr = float(st.sum_wr / max(1e-12, st.sum_w))
            st.last_wr = wr
            st.last_eval_cutoff = cutoff
            n_eval += 1
            eval_wrs.append(wr)

            # update streaks
            if wr <= self.kill_th:
                st.kill_streak += 1
            else:
                st.kill_streak = 0

            if wr >= self.revive_th:
                st.revive_streak += 1
            else:
                st.revive_streak = 0

            prev = float(st.mult)
            new = prev

            if prev >= 0.999 and st.kill_streak >= self.kill_streak_req:
                new = self.throttle_mult
                st.revive_streak = 0
                self.events.append(GateEvent(
                    date=str(trade_day.date()),
                    key=self._key_str(key),
                    prev_mult=prev,
                    new_mult=new,
                    wr=wr,
                    n_trades=n_tr,
                    n_days=n_d,
                    kill_streak=st.kill_streak,
                    revive_streak=st.revive_streak,
                    reason="THROTTLE",
                ))
                n_events += 1

            if prev < 0.999 and st.revive_streak >= self.revive_streak_req:
                new = 1.0
                st.kill_streak = 0
                self.events.append(GateEvent(
                    date=str(trade_day.date()),
                    key=self._key_str(key),
                    prev_mult=prev,
                    new_mult=new,
                    wr=wr,
                    n_trades=n_tr,
                    n_days=n_d,
                    kill_streak=st.kill_streak,
                    revive_streak=st.revive_streak,
                    reason="UNTHROTTLE",
                ))
                n_events += 1

            st.mult = float(new)
            if st.mult < 0.999:
                n_throttled += 1

        # daily aggregate snapshot
        if eval_wrs:
            arr = np.array(eval_wrs, dtype=float)
            row = dict(
                trade_day=str(trade_day.date()),
                cutoff_day=str(cutoff.date()),
                keys_total=int(n_keys),
                keys_evaluated=int(n_eval),
                keys_throttled=int(n_throttled),
                wr_min=float(np.min(arr)),
                wr_p25=float(np.quantile(arr, 0.25)),
                wr_median=float(np.median(arr)),
                wr_p75=float(np.quantile(arr, 0.75)),
                wr_max=float(np.max(arr)),
                exits_seen_total=int(self._exits_seen),
                events_today=int(n_events),
            )
        else:
            row = dict(
                trade_day=str(trade_day.date()),
                cutoff_day=str(cutoff.date()),
                keys_total=int(n_keys),
                keys_evaluated=0,
                keys_throttled=int(n_throttled),
                wr_min=np.nan, wr_p25=np.nan, wr_median=np.nan, wr_p75=np.nan, wr_max=np.nan,
                exits_seen_total=int(self._exits_seen),
                events_today=int(n_events),
            )
        self.daily_rows.append(row)

    def write_outputs(self, out_dir: str):
        os.makedirs(out_dir, exist_ok=True)
        pd.DataFrame(self.daily_rows).to_csv(os.path.join(out_dir, f"wr_gate_daily_{self.name}.csv"), index=False)
        if self.events:
            pd.DataFrame([e.__dict__ for e in self.events]).to_csv(os.path.join(out_dir, f"wr_gate_events_{self.name}.csv"), index=False)
        else:
            pd.DataFrame(columns=list(GateEvent.__annotations__.keys())).to_csv(os.path.join(out_dir, f"wr_gate_events_{self.name}.csv"), index=False)

        # end-state
        rows = []
        for k, st in self.states.items():
            rows.append(dict(
                key=self._key_str(k),
                mult=float(st.mult),
                kill_streak=int(st.kill_streak),
                revive_streak=int(st.revive_streak),
                last_wr=float(st.last_wr) if np.isfinite(st.last_wr) else np.nan,
                last_eval_cutoff=str(st.last_eval_cutoff.date()) if st.last_eval_cutoff is not None else "",
                window_trades=int(len(st.dq)),
                window_days=int(len(st.day_counts)),
                sum_w=float(st.sum_w),
                sum_wr=float(st.sum_wr),
            ))
        pd.DataFrame(rows).to_csv(os.path.join(out_dir, f"wr_gate_state_end_{self.name}.csv"), index=False)


# =========================================================
# Finite capital simulator + TRADE LEDGER (same-day exits fixed) + throttle gate
# =========================================================
@dataclass
class OpenPos:
    pos_id: int
    entry_date: pd.Timestamp
    planned_exit_date: pd.Timestamp
    symbol: str
    regime: str
    side: str
    side_mode: str
    stop_r: float
    tgt_r: float
    rmult_raw: float
    risk_amt: float
    entry_equity: float
    score: float
    tag: str
    gate_mult: float

def _daily_capacity(equity: float, open_positions: List[OpenPos]) -> Tuple[int, float]:
    slots = max(0, int(MAX_OPEN_POSITIONS - len(open_positions)))
    open_risk = float(sum(p.risk_amt for p in open_positions))
    max_risk_amt = float(MAX_GROSS_RISK_PCT * equity)
    rem_risk_amt = max(0.0, max_risk_amt - open_risk)

    # Conservative capacity: based on BASE risk per trade (not throttled risk)
    r_per = float(RISK_PCT_PER_TRADE * equity)
    by_risk = int(rem_risk_amt // max(1e-9, r_per))
    cap = int(min(MAX_NEW_ENTRIES_PER_DAY, slots, by_risk))
    return cap, open_risk / max(1e-9, equity)

def _pf_key_from_pos(p: OpenPos) -> PFKey:
    return (str(p.regime), str(p.side), str(p.side_mode), float(p.stop_r), float(p.tgt_r))

def _pf_key_from_row(regime: str, side: str, side_mode: str, stop_r: float, tgt_r: float) -> PFKey:
    return (str(regime), str(side), str(side_mode), float(stop_r), float(tgt_r))

def _update_exits_conditional(
    d: pd.Timestamp,
    equity: float,
    open_positions: List[OpenPos],
    outliers: List[dict],
    ledger: List[dict],
    mode: str,  # "pre" (< d) or "same" (== d)
    gate: Optional[RollingWRThrottleGate],
) -> Tuple[float, List[OpenPos], int, float]:
    d = pd.Timestamp(d).normalize()
    if mode == "pre":
        exiting = [p for p in open_positions if p.planned_exit_date < d]
        remaining = [p for p in open_positions if p.planned_exit_date >= d]
    elif mode == "same":
        exiting = [p for p in open_positions if p.planned_exit_date == d]
        remaining = [p for p in open_positions if p.planned_exit_date != d]
    else:
        raise ValueError("mode must be 'pre' or 'same'")

    pnl_total = 0.0
    n_exit = 0
    eq_running = float(equity)

    for p in exiting:
        r_used = float(np.clip(p.rmult_raw, -R_ABS_MAX_SANITY, R_ABS_MAX_SANITY))
        if abs(p.rmult_raw) > R_ABS_MAX_SANITY:
            outliers.append(dict(
                kind="rmult_sanity_clamp",
                date=str(d.date()),
                symbol=p.symbol,
                rmult=p.rmult_raw,
                used_rmult=r_used,
                risk_amt=p.risk_amt,
                tag=p.tag,
                pos_id=p.pos_id
            ))

        eq_before = eq_running
        pnl = float(p.risk_amt * r_used)
        eq_after = eq_before + pnl

        # Use planned exit day for attribution (causal); in "pre" mode, planned_exit_date < d
        exit_day_attr = p.planned_exit_date if mode == "pre" else d
        exit_day_attr = pd.Timestamp(exit_day_attr).normalize()

        if gate is not None:
            gate.add_exit(_pf_key_from_pos(p), exit_day_attr, r_used=r_used, risk_amt=p.risk_amt)

        ledger.append(dict(
            tag=p.tag,
            pos_id=int(p.pos_id),
            symbol=str(p.symbol),
            regime=str(p.regime),
            side=str(p.side),
            side_mode=str(p.side_mode),
            stop_r=float(p.stop_r),
            tgt_r=float(p.tgt_r),
            entry_date=str(p.entry_date.date()),
            planned_exit_date=str(p.planned_exit_date.date()),
            exit_date=str(exit_day_attr.date()),
            hold_days=int((exit_day_attr - pd.Timestamp(p.entry_date).normalize()).days),
            score=float(p.score),
            entry_equity=float(p.entry_equity),
            risk_amt=float(p.risk_amt),
            gate_mult=float(p.gate_mult),
            rmult_raw=float(p.rmult_raw),
            rmult_used=float(r_used),
            pnl=float(pnl),
            equity_before_exit=float(eq_before),
            equity_after_exit=float(eq_after),
            exit_mode=str(mode),
        ))

        eq_running = eq_after
        pnl_total += pnl
        n_exit += 1

    return float(eq_running), remaining, int(n_exit), float(pnl_total)

def _choose_one_rr_per_signal(df_day: pd.DataFrame, rr_score_col: str) -> pd.DataFrame:
    if not ENFORCE_ONE_RR_PER_SIGNAL:
        return df_day
    df = df_day.copy()
    df["_tiebreak_rr"] = (df["tgt_r"].astype(float) / np.maximum(1e-6, df["stop_r"].astype(float))).astype(float)

    sort_cols = [rr_score_col]
    asc = [False]
    if "baseline_cagr" in df.columns:
        sort_cols.append("baseline_cagr")
        asc.append(False)
    sort_cols.append("_tiebreak_rr")
    asc.append(False)

    df = df.sort_values(sort_cols, ascending=asc)
    out = df.groupby(["symbol", "entry_date", "side"], sort=False).head(1).drop(columns=["_tiebreak_rr"])
    return out

def _apply_winner_bonus(df_day: pd.DataFrame, pf_cov_map: Dict[Tuple, float]) -> np.ndarray:
    bonus = np.zeros((len(df_day),), dtype=np.float32)
    if df_day.empty:
        return bonus
    keys = list(zip(
        df_day["regime"],
        df_day["side"],
        df_day["side_mode"],
        df_day["stop_r"].astype(float),
        df_day["tgt_r"].astype(float),
    ))
    for i, k in enumerate(keys):
        cov = float(pf_cov_map.get(k, 1.0))
        if cov < WINNER_COVERAGE_FLOOR:
            bonus[i] = float(WINNER_UNDERCOVER_BONUS)
    return bonus

def simulate_fold_global(
    dates: List[pd.Timestamp],
    fold_test: pd.DataFrame,
    signal_score_col: str,
    rr_score_col: str,
    equity0: float,
    open_positions0: List[OpenPos],
    outliers: List[dict],
    ledger: List[dict],
    tag: str,
    pos_counter0: int,
    gate: Optional[RollingWRThrottleGate],
    gate_enabled: bool,
) -> Tuple[pd.DataFrame, float, List[OpenPos], int]:

    equity = float(equity0)
    open_positions = list(open_positions0)
    pos_counter = int(pos_counter0)

    pf_total = fold_test.groupby(["regime","side","side_mode","stop_r","tgt_r"], sort=False).size().to_dict()
    pf_taken = {k: 0 for k in pf_total.keys()}

    daily_rows = []
    by_day = {d: g.copy() for d, g in fold_test.groupby("entry_date", sort=False)}

    for d in dates:
        d = pd.Timestamp(d).normalize()

        # PRE-ENTRY exits (planned_exit_date < d)
        equity, open_positions, n_exit_pre, pnl_exit_pre = _update_exits_conditional(
            d, equity, open_positions, outliers, ledger, mode="pre", gate=gate if gate_enabled else None
        )

        # Gate update: decisions for day d use performance up to (d-1)
        if gate_enabled and gate is not None:
            gate.update_states_for_cutoff(d)

        cap, open_risk_pct_pre = _daily_capacity(equity, open_positions)

        g = by_day.get(d)
        n_cand = 0
        n_sig = 0
        n_take = 0
        avg_score = np.nan
        avg_gate_mult = np.nan
        throttled_taken = 0

        if g is not None and not g.empty and cap > 0:
            df_day = g.copy()
            n_cand = int(len(df_day))

            # pick RR per signal
            dup_before = int(df_day.duplicated(["symbol", "entry_date", "side"], keep=False).sum())
            df_sig = _choose_one_rr_per_signal(df_day, rr_score_col=rr_score_col)
            dup_after = int(df_sig.duplicated(["symbol", "entry_date", "side"], keep=False).sum())
            n_sig = int(len(df_sig))
            if dup_before > 0 or dup_after > 0:
                log.info(f"[SIM {tag}][{d.date()}] signal_dup_before={dup_before} signal_dup_after={dup_after} n_cand={len(df_day):,} n_sig={n_sig:,}")

            if n_sig > 0:
                pf_cov_map = {}
                for k in pf_total.keys():
                    denom = max(1, pf_total.get(k, 0))
                    pf_cov_map[k] = float(pf_taken.get(k, 0) / denom)

                if signal_score_col == "expr_score":
                    bonus = _apply_winner_bonus(df_sig, pf_cov_map)
                    df_sig["_signal_score_adj"] = df_sig[signal_score_col].astype(np.float32) + bonus
                    use_signal = "_signal_score_adj"
                else:
                    use_signal = signal_score_col

                # Optional penalty for throttled keys (ranking only)
                if gate_enabled and gate is not None and WR_GATE_SCORE_PENALTY > 0:
                    penalty = float(WR_GATE_SCORE_PENALTY)
                    mults = []
                    for r in df_sig.itertuples(index=False):
                        k = _pf_key_from_row(r.regime, r.side, r.side_mode, r.stop_r, r.tgt_r)
                        mults.append(gate.get_mult(k))
                    mults = np.array(mults, dtype=np.float32)
                    df_sig["_gate_mult"] = mults
                    df_sig[use_signal] = df_sig[use_signal].astype(np.float32) - np.where(mults < 0.999, penalty, 0.0).astype(np.float32)

                reserved_total = int(round(WINNER_RESERVE_FRAC * cap))
                if reserved_total > 0:
                    cnt_pf = df_sig.groupby(["regime","side_mode","stop_r","tgt_r"], sort=False).size().to_dict()
                    sum_cnt = float(sum(cnt_pf.values())) if cnt_pf else 0.0
                    reserved_pf = {}
                    if sum_cnt > 0:
                        for k, c in cnt_pf.items():
                            reserved_pf[k] = int(round(reserved_total * (float(c) / sum_cnt)))

                    picked_idx = []
                    used = 0
                    if reserved_pf:
                        df_sig2 = df_sig.sort_values(use_signal, ascending=False)
                        for k, kslots in reserved_pf.items():
                            if kslots <= 0:
                                continue
                            dfk = df_sig2[
                                (df_sig2["regime"] == k[0]) &
                                (df_sig2["side_mode"] == k[1]) &
                                (df_sig2["stop_r"] == k[2]) &
                                (df_sig2["tgt_r"] == k[3])
                            ]
                            if dfk.empty:
                                continue
                            take_k = min(kslots, len(dfk), cap - used)
                            if take_k <= 0:
                                continue
                            picked_idx.extend(dfk.head(take_k).index.tolist())
                            used += take_k
                            if used >= cap:
                                break

                    picked = df_sig.loc[picked_idx].copy() if picked_idx else df_sig.iloc[0:0].copy()
                    rem = cap - len(picked)
                    if rem > 0:
                        rest = df_sig.drop(index=picked.index, errors="ignore").sort_values(use_signal, ascending=False)
                        picked = pd.concat([picked, rest.head(rem)], axis=0)
                    df_take = picked.copy()
                else:
                    df_take = df_sig.sort_values(use_signal, ascending=False).head(cap)

                if SELECTION_MODE == "expr_threshold":
                    df_take = df_take[df_take[signal_score_col].astype(float) >= float(EXPR_THRESHOLD)].copy()

                if not df_take.empty:
                    avg_score = float(df_take[signal_score_col].astype(float).mean())

                    # Open new positions, applying throttle to risk per PFKey
                    gate_mults_taken = []
                    for r in df_take.itertuples(index=False):
                        base_risk_amt = float(RISK_PCT_PER_TRADE * equity)
                        gm = 1.0
                        if gate_enabled and gate is not None:
                            k = _pf_key_from_row(r.regime, r.side, r.side_mode, r.stop_r, r.tgt_r)
                            gm = float(gate.get_mult(k))
                        risk_amt = float(base_risk_amt * gm)
                        pos_counter += 1
                        open_positions.append(OpenPos(
                            pos_id=int(pos_counter),
                            entry_date=pd.Timestamp(r.entry_date).normalize(),
                            planned_exit_date=pd.Timestamp(r.exit_date).normalize(),
                            symbol=str(r.symbol),
                            regime=str(r.regime),
                            side=str(r.side),
                            side_mode=str(r.side_mode),
                            stop_r=float(r.stop_r),
                            tgt_r=float(r.tgt_r),
                            rmult_raw=float(r.rmult),
                            risk_amt=float(risk_amt),
                            entry_equity=float(equity),
                            score=float(getattr(r, signal_score_col)),
                            tag=str(tag),
                            gate_mult=float(gm),
                        ))
                        gate_mults_taken.append(gm)
                        if gm < 0.999:
                            throttled_taken += 1

                        pfk = (r.regime, r.side, r.side_mode, float(r.stop_r), float(r.tgt_r))
                        if pfk in pf_taken:
                            pf_taken[pfk] += 1

                    n_take = int(len(df_take))
                    if gate_mults_taken:
                        avg_gate_mult = float(np.mean(np.asarray(gate_mults_taken, dtype=float)))

        # POST-ENTRY same-day exits (planned_exit_date == d) — do NOT influence today's gate decision
        equity, open_positions, n_exit_same, pnl_exit_same = _update_exits_conditional(
            d, equity, open_positions, outliers, ledger, mode="same", gate=gate if gate_enabled else None
        )

        open_risk_pct_end = float(sum(p.risk_amt for p in open_positions)) / max(1e-9, float(equity))

        daily_rows.append(dict(
            tag=tag,
            date_dt=d,
            date=str(d.date()),
            equity=float(equity),
            open_positions=int(len(open_positions)),
            open_risk_pct=float(open_risk_pct_end),
            cap_entries=int(cap),
            n_candidates=int(n_cand),
            n_unique_signals=int(n_sig),
            n_entries=int(n_take),
            dup_cand_same_signal=int(dup_before) if g is not None and not g.empty else 0,
            n_entries_throttled=int(throttled_taken),
            avg_entry_gate_mult=float(avg_gate_mult) if pd.notna(avg_gate_mult) else np.nan,
            n_exits=int(n_exit_pre + n_exit_same),
            pnl_exits=float(pnl_exit_pre + pnl_exit_same),
            avg_score=avg_score,
            exits_pre=int(n_exit_pre),
            exits_same_day=int(n_exit_same),
            open_risk_pct_pre=float(open_risk_pct_pre),
        ))

    out = pd.DataFrame(daily_rows)
    assert_day_dt(out, "date_dt", name="[DAILY] ")
    return out, float(equity), open_positions, int(pos_counter)


def open_positions_terminal_mark_df(ops: List[OpenPos], label_date: pd.Timestamp) -> pd.DataFrame:
    rows = []
    ld = pd.Timestamp(label_date).normalize()
    for p in ops:
        r_used = float(np.clip(p.rmult_raw, -R_ABS_MAX_SANITY, R_ABS_MAX_SANITY))
        rows.append(dict(
            tag=p.tag,
            pos_id=p.pos_id,
            symbol=p.symbol,
            regime=p.regime,
            side=p.side,
            side_mode=p.side_mode,
            stop_r=p.stop_r,
            tgt_r=p.tgt_r,
            entry_date=str(p.entry_date.date()),
            planned_exit_date=str(p.planned_exit_date.date()),
            report_date=str(ld.date()),
            rmult_raw=p.rmult_raw,
            rmult_used=r_used,
            risk_amt=p.risk_amt,
            terminal_source_pnl=float(p.risk_amt * r_used),
            gate_mult=p.gate_mult,
            score=p.score,
        ))
    return pd.DataFrame(rows)

def reindex_equity_to_business_days(eq_df: pd.DataFrame, start_ts: pd.Timestamp, end_ts: pd.Timestamp) -> pd.DataFrame:
    if eq_df.empty:
        return eq_df.copy()
    s = pd.Timestamp(start_ts).normalize()
    e = pd.Timestamp(end_ts).normalize()
    idx = pd.date_range(s, e, freq="B")
    x = eq_df.copy().sort_values("date_dt").set_index("date_dt")
    keep_cols = [c for c in x.columns if c != "date"]
    x = x[keep_cols].reindex(idx)
    # event-like columns that should zero-fill if absent
    zero_cols = [c for c in ["n_exits", "pnl_exits", "n_entries", "n_entries_throttled", "n_candidates", "n_unique_signals", "exits_pre", "exits_same_day"] if c in x.columns]
    ffill_cols = [c for c in x.columns if c not in zero_cols]
    x[ffill_cols] = x[ffill_cols].ffill()
    for c in zero_cols:
        x[c] = x[c].fillna(0)
    x = x.reset_index().rename(columns={"index": "date_dt"})
    x["date"] = x["date_dt"].dt.date.astype(str)
    return x

# =========================================================
# MAIN
# =========================================================
def main():
    t0 = time.time()

    eval_start = pd.Timestamp(EVAL_START).normalize()
    eval_end = pd.Timestamp(EVAL_END).normalize()

    debug_counts = []
    log.info(f"[OUT_DIR] {OUT_DIR}")
    log.info(f"[EVAL] {eval_start.date()}..{eval_end.date()}")
    log.info(f"[FINITE] start_equity={START_EQUITY} risk_pct={RISK_PCT_PER_TRADE} max_open_pos={MAX_OPEN_POSITIONS} max_gross_risk_pct={MAX_GROSS_RISK_PCT} max_new_entries_day={MAX_NEW_ENTRIES_PER_DAY}")
    log.info(f"[RR_POLICY] enable={RR_POLICY_ENABLE} bucket_aware={RR_POLICY_BUCKET_AWARE} global_fallback={RR_POLICY_GLOBAL_FALLBACK} w_baseline_cagr={RR_POLICY_BASELINE_CAGR_WEIGHT} ridge_alpha={RR_POLICY_RIDGE_ALPHA} solver={RR_POLICY_RIDGE_SOLVER}")
    log.info(f"[MISSING_BUCKET] fallback={MISSING_BUCKET_FALLBACK}")
    log.info(f"[WR_GATE] enable_model={WR_GATE_ENABLE_MODEL} window_days={WR_GATE_WINDOW_DAYS} min_trades={WR_GATE_MIN_TRADES} min_days={WR_GATE_MIN_DAYS} kill_th={WR_GATE_KILL_TH} revive_th={WR_GATE_REVIVE_TH} kill_streak={WR_GATE_KILL_STREAK} revive_streak={WR_GATE_REVIVE_STREAK} throttle_mult={WR_GATE_THROTTLE_MULT} score_penalty={WR_GATE_SCORE_PENALTY}")

    baseline_extract = os.path.join(OUT_DIR, "_baseline_extracted")
    root = resolve_root_dir(BASELINE_RESULTS_PATH, baseline_extract)
    pr = locate_portfolio_run_dir(root)
    trades_dir = locate_trades_dir(pr)
    equity_dir = locate_equity_dir(pr)

    eq_map = load_baseline_equity_curves(equity_dir)
    trades = load_all_trades_from_dir(trades_dir)
    debug_counts.append(dict(step="loaded_trades", rows=int(len(trades)), symbols=int(trades["symbol"].nunique()) if len(trades) else 0))

    trades = trades[trades["regime"].isin(["bull","bear"])].copy()
    trades = trades[trades["side"].isin(["long","short"])].copy()
    trades = trades[trades["side_mode"].isin(["both","long","short"])].copy()
    trades = trades[(trades["entry_date"] >= eval_start) & (trades["entry_date"] <= eval_end)].copy()
    if trades.empty:
        raise RuntimeError("No trades after filters.")
    log.info(f"[TRADES] after filters rows={len(trades):,} symbols={trades['symbol'].nunique():,}")
    debug_counts.append(dict(step="after_trade_filters", rows=int(len(trades)), symbols=int(trades["symbol"].nunique()) if len(trades) else 0))

    symbols_needed = sorted(trades["symbol"].unique().tolist())
    series_map, seg_map = load_ohlcv_subset_build_series(OHLCV_FLAGS_CSV, symbols_needed)

    trades = apply_gap_skip(trades, seg_map)
    forced_skips = trades[trades["gap_skip"] == 1].copy()
    trades = trades[trades["gap_skip"] == 0].copy()
    log.info(f"[GAP] forced_skips={len(forced_skips):,} kept_trades={len(trades):,}")
    debug_counts.append(dict(step="after_gap_skip", rows=int(len(trades)), symbols=int(trades["symbol"].nunique()) if len(trades) else 0, forced_skips=int(len(forced_skips))))
    if trades.empty:
        raise RuntimeError("No trades left after gap skip.")

    data = trades.copy()
    data["y_win"] = (pd.to_numeric(data["rmult"], errors="coerce") > 0).astype(int)
    data["stop_r"] = pd.to_numeric(data["stop_atr"], errors="coerce").round(6)
    data["tgt_r"]  = pd.to_numeric(data["target_atr"], errors="coerce").round(6)

    assert_day_dt(data, "entry_date", name="[DATA] ")
    assert_day_dt(data, "exit_date",  name="[DATA] ")

    dedup_cols = ["symbol","entry_date","exit_date","regime","side","side_mode","stop_r","tgt_r","rmult"]
    before = len(data)
    data = data.drop_duplicates(subset=dedup_cols, keep="first").copy()
    log.info(f"[DEDUP] before={before:,} after={len(data):,} removed={before-len(data):,}")
    debug_counts.append(dict(step="after_row_dedup", rows=int(len(data)), symbols=int(data["symbol"].nunique()) if len(data) else 0, removed=int(before-len(data))))

    data = data.sort_values(["entry_date","symbol"]).reset_index(drop=True)
    data["row_id"] = np.arange(len(data), dtype=np.int64)

    # Static features
    static_path = None
    static_dim = 0
    if ADD_STATIC_FEATURES:
        data["stop_atr_f"] = pd.to_numeric(data["stop_atr"], errors="coerce").fillna(0.0).astype(np.float32)
        data["target_atr_f"] = pd.to_numeric(data["target_atr"], errors="coerce").fillna(0.0).astype(np.float32)
        data["rr_ratio"] = (data["target_atr_f"] / np.maximum(1e-6, data["stop_atr_f"])).astype(np.float32)
        data["rr_diff"]  = (data["target_atr_f"] - data["stop_atr_f"]).astype(np.float32)
        data["side_is_long"] = (data["side"] == "long").astype(np.float32)

        base_cols = ["stop_atr_f","target_atr_f","rr_ratio","rr_diff"]
        cs_cols = add_cs_z_rank_inplace(data, cols=base_cols, group_col="entry_date")
        static_cols = base_cols + cs_cols + ["side_is_long"]
        static_dim = len(static_cols)

        static_path = os.path.join(OUT_DIR, "features", "features_static_f16.mmap")
        if not os.path.exists(static_path):
            mm_static_w = np.memmap(static_path, dtype=FEATURE_DTYPE, mode="w+", shape=(len(data), static_dim))
            mm_static_w[:, :] = data[static_cols].replace([np.inf,-np.inf], np.nan).fillna(0.0).astype(np.float32).values.astype(FEATURE_DTYPE)
            mm_static_w.flush()
            del mm_static_w
        log.info(f"[STATIC] path={static_path} shape=({len(data):,},{static_dim})")

    # Folds
    folds = build_yearly_folds(eval_start, eval_end, ROLLING_LOOKBACK_YEARS, ROLLING_FIRST_TEST_YEAR)
    if not folds:
        raise RuntimeError("No folds generated.")
    pd.DataFrame(folds).to_csv(os.path.join(OUT_DIR, "folds", "folds_table.csv"), index=False)
    log.info(f"[FOLDS] count={len(folds)} years={folds[0]['fold_year']}..{folds[-1]['fold_year']}")

    # ROCKET memmap
    rocket_cfg = normalize_rocket_cfg(ROCKET_CFG)
    rocket_dim = rocket_feature_dim(rocket_cfg)
    rocket_path = os.path.join(OUT_DIR, "features", "features_rocket_f16.mmap")
    meta_path = os.path.join(OUT_DIR, "features", "rocket_meta.json")
    ok_path = os.path.join(OUT_DIR, "features", "rocket_ok_all.npy")

    need_build = True
    if os.path.exists(rocket_path) and os.path.exists(meta_path) and os.path.exists(ok_path):
        try:
            meta = json.load(open(meta_path, "r"))
            if meta.get("n_rows") == int(len(data)) and meta.get("rocket_cfg") == rocket_cfg and int(meta.get("rocket_dim", -1)) == int(rocket_dim):
                need_build = False
        except Exception:
            need_build = True

    if need_build:
        log.info(f"[ROCKET] building memmap cfg={rocket_cfg} dim={rocket_dim}")
        _, ok_all = write_rocket_memmap(data, series_map, rocket_cfg, seed=ROCKET_RANDOM_SEED, out_path=rocket_path, batch_size=ROCKET_BATCH_SIZE)
        np.save(ok_path, ok_all.astype(np.uint8))
        json.dump(dict(n_rows=int(len(data)), rocket_cfg=rocket_cfg, rocket_dim=int(rocket_dim), static_dim=int(static_dim)),
                  open(meta_path, "w"), indent=2)
        data["rocket_ok_all"] = ok_all.astype(int)
    else:
        log.info("[ROCKET] reusing existing memmap/meta/ok_all")
        ok_all = np.load(ok_path).astype(np.uint8)
        if len(ok_all) != len(data):
            raise RuntimeError(f"rocket_ok_all length mismatch: {len(ok_all)} vs {len(data)}")
        data["rocket_ok_all"] = ok_all.astype(int)

    mm_rocket = open_memmap(rocket_path, shape=(len(data), rocket_dim))
    mm_static = open_memmap(static_path, shape=(len(data), static_dim)) if (ADD_STATIC_FEATURES and static_path) else None

    # Gates
    gate_model = RollingWRThrottleGate("model") if WR_GATE_ENABLE_MODEL else None
    gate_base = RollingWRThrottleGate("baseline") if WR_GATE_ENABLE_BASELINE else None

    equity_model = float(START_EQUITY)
    equity_base = float(START_EQUITY)
    open_model: List[OpenPos] = []
    open_base: List[OpenPos] = []

    daily_all_model = []
    daily_all_base = []
    fold_stats_rows = []
    outliers = []

    ledger_base: List[dict] = []
    ledger_model: List[dict] = []
    pos_counter_base = 0
    pos_counter_model = 0

    cal = pd.concat([data["entry_date"], data["exit_date"]], ignore_index=True).dropna()
    all_calendar_dates = pd.DatetimeIndex(pd.unique(cal.values)).sort_values().normalize()
    log.info(f"[CAL] unique_dates={len(all_calendar_dates):,} min={all_calendar_dates.min().date()} max={all_calendar_dates.max().date()}")

    for f in folds:
        fold_year = f["fold_year"]
        fold_tag = str(fold_year)
        train_start = pd.Timestamp(f["train_start"]).normalize()
        train_end   = pd.Timestamp(f["train_end"]).normalize()
        test_start  = pd.Timestamp(f["test_start"]).normalize()
        test_end    = pd.Timestamp(f["test_end"]).normalize()

        log.info(f"[FOLD {fold_tag}] train={train_start.date()}..{train_end.date()} test={test_start.date()}..{test_end.date()}")

        winners_tbl, winners_df = select_winners(eq_map, data, train_start, train_end)
        winners_tbl.to_csv(os.path.join(OUT_DIR, "folds", f"fold_{fold_tag}_winners_table.csv"), index=False)
        log.info(f"[FOLD {fold_tag}] winners={len(winners_df):,}")
        if winners_df.empty:
            continue

        winners_keys = winners_df[["regime","side_mode","stop_r","tgt_r","baseline_cagr"]].copy()
        winners_keys["stop_r"] = pd.to_numeric(winners_keys["stop_r"], errors="coerce").round(6)
        winners_keys["tgt_r"]  = pd.to_numeric(winners_keys["tgt_r"], errors="coerce").round(6)

        fold_train = data[
            (data["entry_date"] >= train_start) &
            (data["entry_date"] <= train_end) &
            (data["exit_date"]  <= train_end)
        ].merge(winners_keys, on=["regime","side_mode","stop_r","tgt_r"], how="inner")

        fold_test = data[
            (data["entry_date"] >= test_start) &
            (data["entry_date"] <= test_end)
        ].merge(winners_keys, on=["regime","side_mode","stop_r","tgt_r"], how="inner")

        leaked_removed = int((
            (data["entry_date"] >= train_start) &
            (data["entry_date"] <= train_end) &
            (data["exit_date"]  >  train_end)
        ).sum())
        log.info(f"[FOLD {fold_tag}] NOLEAK removed_train_cross_boundary_trades={leaked_removed:,}")
        log.info(f"[FOLD {fold_tag}] fold_train_rows={len(fold_train):,} fold_test_rows={len(fold_test):,}")

        if len(fold_train) < max(20_000, MIN_TRAIN_ROWS) or fold_train["y_win"].nunique() < 2:
            log.warning(f"[FOLD {fold_tag}] too little training; skipping fold.")
            continue
        if fold_test.empty:
            continue

        fold_models_dir = os.path.join(OUT_DIR, "models", f"fold_{fold_tag}")
        bucket_models, mstats = train_fold_bucket_models(fold_tag, fold_train, mm_rocket, mm_static, fold_models_dir)
        mstats.to_csv(os.path.join(OUT_DIR, "folds", f"fold_{fold_tag}_model_stats.csv"), index=False)

        rr_bucket, rr_global, rr_stats = train_rr_policy_models(fold_tag, fold_train, mm_rocket, mm_static, fold_models_dir)
        if not rr_stats.empty:
            rr_stats.to_csv(os.path.join(OUT_DIR, "folds", f"fold_{fold_tag}_rr_policy_stats.csv"), index=False)

        fold_test = fold_test.sort_values(["entry_date","symbol"]).copy()
        fold_test["baseline_score"] = fold_test["baseline_cagr"].astype(np.float32)

        expr_raw, expr_valid = score_expr_for_df(fold_test, bucket_models, mm_rocket, mm_static)
        fold_test["expr_score_raw"] = expr_raw.astype(np.float32)
        fold_test["expr_valid"] = expr_valid.astype(int)

        if MISSING_BUCKET_FALLBACK == "baseline":
            fold_test["expr_score"] = np.where(expr_valid == 1, expr_raw, fold_test["baseline_score"].values.astype(np.float32)).astype(np.float32)
        else:
            fold_test["expr_score"] = np.where(expr_valid == 1, expr_raw, np.nan).astype(np.float32)
            before_drop = len(fold_test)
            fold_test = fold_test[np.isfinite(fold_test["expr_score"])].copy()
            log.warning(f"[FOLD {fold_tag}] dropped_missing_bucket_rows={before_drop-len(fold_test):,}")

        miss = int((expr_valid == 0).sum())
        if miss > 0:
            log.warning(f"[FOLD {fold_tag}] missing bucket scores={miss:,}/{len(expr_valid):,} fallback={MISSING_BUCKET_FALLBACK}")

        fold_test["rr_choice_score_baseline"] = fold_test["baseline_score"].astype(np.float32)
        if RR_POLICY_ENABLE and (not fold_test.empty):
            rr_pred = predict_rr_policy_for_df(fold_test, rr_bucket, rr_global, mm_rocket, mm_static)
            fold_test["rr_pred"] = rr_pred.astype(np.float32)
            base = fold_test["baseline_score"].astype(np.float32).values
            rr_choice = np.where(np.isfinite(rr_pred),
                                 rr_pred.astype(np.float32) + float(RR_POLICY_BASELINE_CAGR_WEIGHT) * base,
                                 base).astype(np.float32)
            fold_test["rr_choice_score_model"] = rr_choice
        else:
            fold_test["rr_choice_score_model"] = fold_test["baseline_score"].astype(np.float32)

        dates = [d for d in all_calendar_dates if (d >= test_start and d <= test_end)]
        if not dates or fold_test.empty:
            continue

        daily_base, equity_base, open_base, pos_counter_base = simulate_fold_global(
            dates=dates,
            fold_test=fold_test,
            signal_score_col="baseline_score",
            rr_score_col="rr_choice_score_baseline",
            equity0=equity_base,
            open_positions0=open_base,
            outliers=outliers,
            ledger=ledger_base,
            tag=f"baseline_{fold_tag}",
            pos_counter0=pos_counter_base,
            gate=gate_base,
            gate_enabled=bool(WR_GATE_ENABLE_BASELINE),
        )

        daily_model, equity_model, open_model, pos_counter_model = simulate_fold_global(
            dates=dates,
            fold_test=fold_test,
            signal_score_col="expr_score",
            rr_score_col="rr_choice_score_model",
            equity0=equity_model,
            open_positions0=open_model,
            outliers=outliers,
            ledger=ledger_model,
            tag=f"model_{fold_tag}",
            pos_counter0=pos_counter_model,
            gate=gate_model,
            gate_enabled=bool(WR_GATE_ENABLE_MODEL),
        )

        daily_all_base.append(daily_base)
        daily_all_model.append(daily_model)

        fold_stats_rows.append(dict(
            fold_year=fold_year,
            train_start=str(train_start.date()), train_end=str(train_end.date()),
            test_start=str(test_start.date()), test_end=str(test_end.date()),
            winners=int(len(winners_df)),
            fold_train_rows=int(len(fold_train)),
            fold_test_rows=int(len(fold_test)),
            end_equity_baseline=float(equity_base),
            end_equity_model=float(equity_model),
            missing_bucket_scores=int(miss),
        ))

        log.info(f"[FOLD {fold_tag}] end_equity baseline={equity_base:,.2f} model={equity_model:,.2f}")

    if not daily_all_base or not daily_all_model:
        raise RuntimeError("No fold simulations produced outputs (all folds skipped?).")

    base_all = pd.concat(daily_all_base, ignore_index=True)
    model_all = pd.concat(daily_all_model, ignore_index=True)

    base_eq = base_all.sort_values(["date_dt"]).groupby("date_dt", as_index=False).tail(1).copy()
    model_eq = model_all.sort_values(["date_dt"]).groupby("date_dt", as_index=False).tail(1).copy()

    def add_drawdown(df):
        eq = df["equity"].astype(float).values
        peak = np.maximum.accumulate(eq)
        dd = (eq / np.maximum(1e-9, peak)) - 1.0
        df["drawdown"] = dd.astype(float)
        return df

    base_eq = add_drawdown(base_eq)
    model_eq = add_drawdown(model_eq)

    # Core outputs
    base_all.to_csv(os.path.join(OUT_DIR, "global", "daily_baseline.csv"), index=False)
    model_all.to_csv(os.path.join(OUT_DIR, "global", "daily_model.csv"), index=False)
    base_eq.to_csv(os.path.join(OUT_DIR, "global", "equity_baseline.csv"), index=False)
    model_eq.to_csv(os.path.join(OUT_DIR, "global", "equity_model.csv"), index=False)
    if REINDEX_EQUITY_TO_BUSINESS_DAYS:
        base_eq_b = reindex_equity_to_business_days(base_eq, eval_start, eval_end)
        model_eq_b = reindex_equity_to_business_days(model_eq, eval_start, eval_end)
        base_eq_b = add_drawdown(base_eq_b)
        model_eq_b = add_drawdown(model_eq_b)
        base_eq_b.to_csv(os.path.join(OUT_DIR, "global", "equity_baseline_bday.csv"), index=False)
        model_eq_b.to_csv(os.path.join(OUT_DIR, "global", "equity_model_bday.csv"), index=False)
    pd.DataFrame(fold_stats_rows).to_csv(os.path.join(OUT_DIR, "global", "fold_stats.csv"), index=False)
    if WRITE_DEBUG_FILTER_COUNTS:
        pd.DataFrame(debug_counts).to_csv(os.path.join(OUT_DIR, "global", "debug_filter_counts.csv"), index=False)

    if outliers:
        pd.DataFrame(outliers).to_csv(os.path.join(OUT_DIR, "global", "debug_top_outliers.csv"), index=False)

    # Ledgers
    led_base_df = pd.DataFrame(ledger_base)
    led_model_df = pd.DataFrame(ledger_model)
    led_all_df = pd.concat([led_base_df, led_model_df], ignore_index=True) if (not led_base_df.empty or not led_model_df.empty) else pd.DataFrame()

    led_base_df.to_csv(os.path.join(OUT_DIR, "global", "trade_ledger_baseline.csv"), index=False)
    led_model_df.to_csv(os.path.join(OUT_DIR, "global", "trade_ledger_model.csv"), index=False)
    led_all_df.to_csv(os.path.join(OUT_DIR, "global", "trade_ledger_all.csv"), index=False)

    # Open positions end
    def open_positions_to_df(ops: List[OpenPos]) -> pd.DataFrame:
        if not ops:
            return pd.DataFrame(columns=[
                "tag","pos_id","symbol","regime","side","side_mode","stop_r","tgt_r",
                "entry_date","planned_exit_date","rmult_raw","risk_amt","entry_equity","score","gate_mult"
            ])
        rows = []
        for p in ops:
            rows.append(dict(
                tag=p.tag,
                pos_id=p.pos_id,
                symbol=p.symbol,
                regime=p.regime,
                side=p.side,
                side_mode=p.side_mode,
                stop_r=p.stop_r,
                tgt_r=p.tgt_r,
                entry_date=str(p.entry_date.date()),
                planned_exit_date=str(p.planned_exit_date.date()),
                rmult_raw=p.rmult_raw,
                risk_amt=p.risk_amt,
                entry_equity=p.entry_equity,
                score=p.score,
                gate_mult=p.gate_mult,
            ))
        return pd.DataFrame(rows)

    open_base_df = open_positions_to_df(open_base)
    open_model_df = open_positions_to_df(open_model)
    open_base_df.to_csv(os.path.join(OUT_DIR, "global", "open_positions_baseline_end.csv"), index=False)
    open_model_df.to_csv(os.path.join(OUT_DIR, "global", "open_positions_model_end.csv"), index=False)

    if WRITE_DEBUG_OPEN_MARKS:
        terminal_base_df = open_positions_terminal_mark_df(open_base, eval_end)
        terminal_model_df = open_positions_terminal_mark_df(open_model, eval_end)
        terminal_base_df.to_csv(os.path.join(OUT_DIR, "global", "open_positions_baseline_terminal_source_mark.csv"), index=False)
        terminal_model_df.to_csv(os.path.join(OUT_DIR, "global", "open_positions_model_terminal_source_mark.csv"), index=False)
    else:
        terminal_base_df = pd.DataFrame()
        terminal_model_df = pd.DataFrame()

    # Gate outputs
    if gate_model is not None and WR_GATE_ENABLE_MODEL:
        gate_model.write_outputs(os.path.join(OUT_DIR, "global"))
    if gate_base is not None and WR_GATE_ENABLE_BASELINE:
        gate_base.write_outputs(os.path.join(OUT_DIR, "global"))

    def summarize(eq_df, name, open_df_terminal: pd.DataFrame):
        e0 = float(eq_df["equity"].iloc[0])
        e1 = float(eq_df["equity"].iloc[-1])
        d0 = pd.Timestamp(eq_df["date_dt"].iloc[0]).normalize()
        d1 = pd.Timestamp(eq_df["date_dt"].iloc[-1]).normalize()
        days = max(1, int((d1 - d0).days))
        cagr = (e1 / e0) ** (365.0 / days) - 1.0 if e0 > 0 and e1 > 0 else np.nan
        mdd = float(eq_df["drawdown"].min())
        term_source_pnl = float(open_df_terminal["terminal_source_pnl"].sum()) if (REPORT_TERMINAL_SOURCE_PNL and not open_df_terminal.empty and "terminal_source_pnl" in open_df_terminal.columns) else 0.0
        closed_book_end = float(e1 + term_source_pnl)
        closed_book_cagr = (closed_book_end / e0) ** (365.0 / days) - 1.0 if e0 > 0 and closed_book_end > 0 else np.nan
        log.info(f"[SUMMARY {name}] realized_start={e0:,.2f} realized_end={e1:,.2f} days={days} CAGR={cagr if pd.notna(cagr) else np.nan} MDD={mdd:.3f} open_count={len(open_df_terminal):,} terminal_source_pnl={term_source_pnl:,.2f} closed_book_end={closed_book_end:,.2f} closed_book_cagr={closed_book_cagr if pd.notna(closed_book_cagr) else np.nan}")
        return dict(name=name, realized_start=e0, realized_end=e1, realized_cagr=float(cagr) if pd.notna(cagr) else np.nan, mdd=mdd, open_count=int(len(open_df_terminal)), terminal_source_pnl=term_source_pnl, closed_book_end=closed_book_end, closed_book_cagr=float(closed_book_cagr) if pd.notna(closed_book_cagr) else np.nan, start_date=str(d0.date()), end_date=str(d1.date()), days=days)

    summ_base = summarize(base_eq, "BASELINE", terminal_base_df)
    summ_model = summarize(model_eq, "MODEL", terminal_model_df)
    pd.DataFrame([summ_base, summ_model]).to_csv(os.path.join(OUT_DIR, "global", "summary_realized_vs_terminal.csv"), index=False)

    log.info(f"[LEDGER] baseline_closed={len(led_base_df):,} model_closed={len(led_model_df):,} all_closed={len(led_all_df):,}")
    log.info(f"[OPEN_END] baseline_open={len(open_base):,} model_open={len(open_model):,}")
    log.info(f"[OUT] {os.path.join(OUT_DIR, 'global')}")
    log.info(f"[DONE] elapsed={time.time()-t0:.1f}s")


if __name__ == "__main__":
    main()

2026-03-22 07:25:31,894 | INFO | [OUT_DIR] /kaggle/working/global_allocator_expr_v3
2026-03-22 07:25:31,896 | INFO | [EVAL] 2015-01-01..2025-12-10
2026-03-22 07:25:31,897 | INFO | [FINITE] start_equity=1000000.0 risk_pct=0.005 max_open_pos=180 max_gross_risk_pct=0.35 max_new_entries_day=120
2026-03-22 07:25:31,898 | INFO | [RR_POLICY] enable=True bucket_aware=True global_fallback=True w_baseline_cagr=0.25 ridge_alpha=100.0 solver=lsqr
2026-03-22 07:25:31,898 | INFO | [MISSING_BUCKET] fallback=baseline
2026-03-22 07:25:31,900 | INFO | [WR_GATE] enable_model=True window_days=60 min_trades=60 min_days=20 kill_th=-0.05 revive_th=0.05 kill_streak=3 revive_streak=5 throttle_mult=0.25 score_penalty=0.0
2026-03-22 07:25:34,472 | INFO | [BASELINE] equity_curves: matched=144 loaded=144
2026-03-22 07:25:34,503 | INFO | [TRADES] Reading parquet files: 144
2026-03-22 07:25:41,973 | INFO | [TRADES] Loaded rows=909,641 symbols=2,278 |R|>20.0 count=96
2026-03-22 07:25:43,587 | INFO | [TRADES] after fi